# Amazon ML Challenge 2026 — Business Entity Resolution (v4.9)

**Kaggle settings:** Accelerator = GPU T4 x2, Internet = ON, challenge dataset attached. Then *Save Version → Save & Run All (Commit)*.

**Pipeline:** normalise → fine-tune multilingual-E5-small (bidirectional in-batch loss, chain siblings as hard negatives)
→ dense top-K + exact-key blocking → XGBoost on similarity features → gte-multilingual-reranker on the ambiguous band
→ uniqueness + expected-F0.5 decoder tuned on an entity-disjoint validation split → TSVs.

A preliminary submission is written as soon as the XGBoost stage finishes, so a later failure still leaves a valid file.

**v4 changes:** clean validation world (no record E5 was trained on; every S1 owner visible to the uniqueness rule),
name-only retrieval channel for empty-address records, name/address frequency features, house-number similarity features.

**Inputs to attach:** the challenge dataset + the Version 1 output (train.parquet, e5.pt, emb_train.npy)
+ the V3.1/V3.2 output (test.parquet, cand_test.parquet, test_p1.npy, pairs_val.parquet, ce_e5ce.pt, ce_test_e5ce_*.npy).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "anyascii", "rapidfuzz>=3.6"], check=False)

In [ ]:
# Config
import os, re, gc, csv, glob, json, time, shutil, unicodedata, traceback
import multiprocessing as mp
from contextlib import contextmanager
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb, transformers
from anyascii import anyascii
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
from rapidfuzz.distance import Levenshtein
import scipy.sparse as sps
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

INPUT = os.environ.get("ER_INPUT", "/kaggle/input")
WORK = os.environ.get("ER_WORK", "/kaggle/working")
E5_MODEL = os.environ.get("ER_E5", "intfloat/multilingual-e5-small")               # MIT, 118M
CE_MODEL = os.environ.get("ER_CE", "Alibaba-NLP/gte-multilingual-reranker-base")   # Apache-2.0, 306M
BUDGET_H = float(os.environ.get("ER_BUDGET_H", 11.4))   # Kaggle kills at 12h; keep a margin
SEED = 42
K = 10                        # dense neighbours (S1) per S2/S3 record
SPLIT = (50, 85)              # S1 hash buckets: <50 E5-train | <85 matcher-train | rest validation
MAXLEN_BI, MAXLEN_CE = 64, 128
E5_PAIRS, E5_BATCH, E5_LR, E5_MAX_MIN = 2_000_000, 256, 3e-5, 90
XGB_MAX_PAIRS = 10_000_000   # 15M + 7 TF-IDF columns ran out of memory in v4.2
CE_PAIRS, CE_BATCH, CE_LR, CE_MAX_MIN = 600_000, 64, 2e-5, 100
CE_MIN_LEFT_H = 4.0           # skip reranker training if less budget than this remains
RESERVE_S = 2700              # budget kept back for final decode + writing files
P_FLOOR = 0.02                # pairs below this are never predicted
ROUTE_FRAC = 0.08             # most-ambiguous share of validation pairs sent to the reranker (sets the band)
KN = 10                       # name-only channel: S1 neighbours per empty-address record
KN_RESCUE = 50                # v4.9: wider name-only search for empty-address records in the rescue
WORLD_DISTRACTOR_PCT = 100    # keep ALL unowned train records: world distractor share ≈41%, like test (~37–40%), not train's 26%

CKPT, OUT, DIAG = f"{WORK}/ckpt", f"{WORK}/output", f"{WORK}/diagnostics"
for d in (CKPT, OUT, DIAG):
    os.makedirs(d, exist_ok=True)
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
NGPU = torch.cuda.device_count()
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")   # T4: fp16, no bf16
T0, RUNTIME, M = time.time(), [], {}    # M collects every metric for summary.json


import ctypes
import pyarrow as pa
try:   # Arrow's own allocator keeps freed parquet buffers (the +3 GB seen in resumed runs); use the system one
    pa.set_memory_pool(pa.system_memory_pool())
except Exception:
    pass
try:
    _LIBC = ctypes.CDLL("libc.so.6")
except OSError:
    _LIBC = None


def trim():
    gc.collect()
    try:
        pa.default_memory_pool().release_unused()
    except Exception:
        pass
    if _LIBC is not None:
        _LIBC.malloc_trim(0)


def log(tag, msg):
    print(f"[{time.strftime('%H:%M:%S', time.gmtime(time.time() - T0))}][{tag}] {msg}", flush=True)


def left():
    return BUDGET_H * 3600 - (time.time() - T0)


RESUME = [d for d in glob.glob(f"{INPUT}/**/ckpt", recursive=True) if os.path.isdir(d)]
_seen, PREFER = {}, {}
for _d in RESUME:   # several previous runs may be attached; only pair-keyed score caches may appear twice
    for _f in os.listdir(_d):
        if _f in _seen:
            if re.match(r"ce_.*cache.*\.npz$", _f) or re.match(r"ce_(test|val)_v\d+_\d+\.npy$", _f):
                big_ = max((_seen[_f], _d), key=lambda d_: os.path.getsize(f"{d_}/{_f}"))
                PREFER[_f] = big_; _seen[_f] = big_   # caches only grow; keep the larger copy
                continue
            raise RuntimeError(f"checkpoint {_f} exists in both {_seen[_f]} and {_d}; attach only one copy")
        _seen[_f] = _d
TMP = os.environ.get("ER_TMP", "/tmp/er")
os.makedirs(TMP, exist_ok=True)


def ck(name, big=False):
    """Existing checkpoint (this run or an attached previous run) if any, else where to write it.
    big=True files go to TMP: re-computable and too large for Kaggle's 20 GB saved-output limit."""
    if name in PREFER and not os.path.exists(f"{CKPT}/{name}"):
        return f"{PREFER[name]}/{name}"
    for d in [CKPT] + RESUME:
        if os.path.exists(f"{d}/{name}"):
            return f"{d}/{name}"
    return f"{TMP if big else CKPT}/{name}"


@contextmanager
def stage(name):
    t = time.time(); log(name, f"start | {left() / 3600:.2f} h budget left")
    yield
    RUNTIME.append((name, round(time.time() - t)))
    pd.DataFrame(RUNTIME, columns=["stage", "seconds"]).to_csv(f"{DIAG}/runtime.csv", index=False)
    trim(); torch.cuda.empty_cache()
    log(name, f"done in {(time.time() - t) / 60:.1f} min | {mem()}")


def mem():
    st = dict(l.split(":", 1) for l in open("/proc/self/status"))
    mi = dict(l.split(":", 1) for l in open("/proc/meminfo"))
    return (f"RSS {int(st['VmRSS'].split()[0]) / 1e6:.1f} GB | RAM available {int(mi['MemAvailable'].split()[0]) / 1e6:.1f} GB"
            f" | disk free {shutil.disk_usage(WORK).free / 1e9:.0f} GB")


def dump():
    json.dump(M, open(f"{DIAG}/summary.json", "w"), indent=1, default=float)


gpus = [torch.cuda.get_device_name(i) for i in range(NGPU)]
log("ENV", f"python {sys.version.split()[0]} | torch {torch.__version__} cuda {torch.version.cuda} | GPUs {gpus}")
log("ENV", f"transformers {transformers.__version__} | xgboost {xgb.__version__} | pandas {pd.__version__}")
log("ENV", f"CPUs {os.cpu_count()} | RAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.0f} GB"
           f" | disk free {shutil.disk_usage(WORK).free / 1e9:.0f} GB")
json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, tuple))},
          open(f"{DIAG}/config.json", "w"), indent=1)

log("ENV", f"tmp disk free {shutil.disk_usage(TMP).free / 1e9:.0f} GB | resuming from {RESUME or 'nothing'}")

In [ ]:
# Discover data, validate schemas, fail fast on the reranker
TSVS = {os.path.basename(p): p for p in glob.glob(f"{INPUT}/**/*.tsv", recursive=True)}
NEED = [f"{sp}_source{s}.tsv" for sp in ("train", "test") for s in (1, 2, 3)] + ["train_ground_truth.tsv"]
missing = [n for n in NEED if n not in TSVS]
if missing:
    raise FileNotFoundError(f"missing {missing} under {INPUT}; found {sorted(TSVS)[:20]}")
VALIDATOR = next(iter(glob.glob(f"{INPUT}/**/validate_submission.py", recursive=True)), None)
COLS = ["entity_id", "business_name", "business_address", "country"]
GT_COLS = ["source1_entity_id", "matched_entity_ids"]


def read_tsv(path, cols, nrows=None):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE, nrows=nrows)
    if set(cols) - set(df.columns):
        raise ValueError(f"{path}: expected columns {cols}, got {list(df.columns)}")
    return df[cols]


for n in NEED:
    read_tsv(TSVS[n], GT_COLS if "ground" in n else COLS, nrows=5)
log("DATA", f"schemas OK | validator: {VALIDATOR}")

def load_ce(name):
    """GTE reranker via its official remote code; fallback = E5 weights + a fresh 1-logit head (standard arch)."""
    remote = name == CE_MODEL
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=remote)
    m = AutoModelForSequenceClassification.from_pretrained(name, trust_remote_code=remote, **({} if remote else {"num_labels": 1}))
    return tok, m


# v4 uses the E5 cross-encoder trained in V3.1 (GTE does not load on transformers 5).

In [ ]:
# Stage 1: multi-view normalisation
# txt: NFKC + casefold, Unicode kept (Devanagari, accents) -> transformers
# nk/ak: transliterated (anyascii), accent-free, ascii tokens -> lexical features and exact keys
# n0..n2: first three distinct numbers of the address (house number, PIN/ZIP)
NONKEY = re.compile(r"[^a-z0-9#]+")


def clean(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().replace("<null>", " ").split())


def keyv(s):
    return " ".join(NONKEY.sub(" ", anyascii(s).lower()).split())


def norm_rows(rows):
    out = []
    for n, a in rows:
        n, a = clean(n), clean(a)
        out.append((f"{n} ; {a}", keyv(n), keyv(a)))
    return out


def load_split(sp):
    path = ck(f"{sp}.parquet")
    if not os.path.exists(path):
        with stage(f"NORMALIZE-{sp}"):
            parts = []
            for s in (1, 2, 3):
                df = read_tsv(TSVS[f"{sp}_source{s}.tsv"], COLS)
                bad = ~df.entity_id.str.startswith(f"S{s}-")
                if bad.any():
                    raise ValueError(f"{sp}_source{s}: ids without S{s}- prefix, e.g. {df.entity_id[bad].head(3).tolist()}")
                parts.append(df.assign(src=np.int8(s)))
                log("NORMALIZE", f"{sp}_source{s}: {len(df):,} rows")
            df = pd.concat(parts, ignore_index=True); del parts
            if df.entity_id.duplicated().any():
                raise ValueError(f"{sp}: duplicate entity_ids")
            rows = list(zip(df.business_name, df.business_address))
            chunks = [rows[i:i + 100_000] for i in range(0, len(rows), 100_000)]
            res = []
            with mp.get_context("fork").Pool(os.cpu_count()) as pool:
                for i, part in enumerate(pool.imap(norm_rows, chunks)):
                    res += part
                    if i % 20 == 0:
                        log("NORMALIZE", f"{len(res):,} / {len(rows):,}")
            del rows, chunks
            txt, nk, ak = map(list, zip(*res)); del res
            nums = pd.Series(ak).str.findall(r"\d+").map(lambda l: list(dict.fromkeys(l))[:3])
            rec = pd.DataFrame({"entity_id": df.entity_id, "src": df.src, "country": df.country.str.strip(),
                                "txt": txt, "nk": nk, "ak": ak})
            for j in range(3):
                rec[f"n{j}"] = nums.map(lambda l: int(l[j][-15:]) + 1 if len(l) > j else 0).astype(np.int64)
            rec.to_parquet(path)
            del df, txt, nk, ak, nums, rec; gc.collect()
    rec = pd.read_parquet(path)
    for c in ("entity_id", "country", "txt", "nk", "ak"):
        rec[c] = rec[c].astype("string[pyarrow]")
    rec["cc"] = pd.factorize(rec.country)[0].astype(np.int16)
    return rec


def take(col, idx):   # memory-light string gather
    return col.array.take(np.asarray(idx)).to_numpy(dtype=object)

In [ ]:
# Ground truth, entity-disjoint split
load_split("test")    # normalise + checkpoint test now, while RAM is still empty
gc.collect()
tr = load_split("train")
N1 = int((tr.src == 1).sum())          # S1 rows are 0..N1-1, S2/S3 rows N1..N-1 (global row ids)
gt = read_tsv(TSVS["train_ground_truth.tsv"], GT_COLS)
truth = gt.assign(rec=gt.matched_entity_ids.str.split(",")).explode("rec")
truth = truth[truth.rec.fillna("").str.strip() != ""]
idx = pd.Index(tr.entity_id.astype(object))
t_si = idx.get_indexer(truth.source1_entity_id).astype(np.int64)
t_qi = idx.get_indexer(truth.rec.str.strip()).astype(np.int64)
assert (t_si >= 0).all() and (t_qi >= 0).all(), "ground-truth ids missing from the train files"
assert (t_si < N1).all() and (t_qi >= N1).all(), "ground truth links must be S1 -> S2/S3"
del gt, truth, idx
n_true = np.bincount(t_si, minlength=N1)
part = pd.util.hash_pandas_object(tr.entity_id[:N1], index=False).to_numpy() % 100
part = np.where(part < SPLIT[0], 0, np.where(part < SPLIT[1], 1, 2)).astype(np.int8)
cc = tr.cc.to_numpy()
M["gt"] = dict(links=len(t_si), s1=N1, singleton_rate=float((n_true == 0).mean()),
               q_linked_to_multiple_s1=int(pd.Series(t_qi).duplicated().sum()),
               cross_country_links=float((cc[t_si] != cc[t_qi]).mean()))
PER_COUNTRY = M["gt"]["cross_country_links"] < 0.005   # partitioned retrieval only if labels agree
log("GT", f"{M['gt']} | per-country retrieval: {PER_COUNTRY}")
dump()

In [ ]:
# Stage 2: fine-tune multilingual-E5-small
class Pooled(nn.Module):   # returns a tensor, so DataParallel can gather it
    def __init__(s, m):
        super().__init__(); s.m = m

    def forward(s, input_ids, attention_mask):
        h = s.m(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        w = attention_mask.unsqueeze(-1).to(h.dtype)
        return F.normalize((h * w).sum(1) / w.sum(1).clamp(min=1e-6), dim=-1)


class Logit(nn.Module):
    def __init__(s, m):
        super().__init__(); s.m = m

    def forward(s, input_ids, attention_mask):
        return s.m(input_ids=input_ids, attention_mask=attention_mask).logits.view(-1)


def par(m):
    m = m.to(DEV)
    return nn.DataParallel(m) if NGPU > 1 else m


def toks(tok, a, b=None, maxlen=MAXLEN_BI):
    enc = tok(list(a), None if b is None else list(b), padding=True, truncation=True, max_length=maxlen, return_tensors="pt")
    return dict(input_ids=enc["input_ids"].to(DEV), attention_mask=enc["attention_mask"].to(DEV))


def train_loop(tag, model, batches, loss_fn, lr, max_min):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.05 * len(batches)), len(batches))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    model.train(); t = time.time(); run = 0.0
    for i, b in enumerate(batches):
        with torch.autocast(**AMP):
            loss = loss_fn(b)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        run = 0.98 * run + 0.02 * loss.item() if i else loss.item()
        if i % 200 == 0 or i == len(batches) - 1:
            el = time.time() - t
            log(tag, f"step {i + 1:,}/{len(batches):,} | loss {run:.4f} | {el / 60:.1f} min | ETA {el / (i + 1) * (len(batches) - i - 1) / 60:.0f} min")
        if time.time() - t > max_min * 60:
            log(tag, f"time cap hit at step {i + 1:,}; stopping early"); break
    model.eval()
    M[f"{tag}_final_loss"] = run


e5_tok = AutoTokenizer.from_pretrained(E5_MODEL)
e5 = AutoModel.from_pretrained(E5_MODEL)
if os.path.exists(ck("e5.pt")):
    e5.load_state_dict(torch.load(ck("e5.pt"), map_location="cpu"))
else:
    with stage("E5-TRAIN"):
        sel = np.flatnonzero(part[t_si] == 0)
        sel = rng.permutation(sel)[:E5_PAIRS]
        a_si, b_qi = t_si[sel], t_qi[sel]
        # chain siblings (same normalised name) land in the same batch -> in-batch hard negatives
        order = pd.DataFrame({"k": take(tr.nk, a_si), "r": rng.random(len(sel))}).sort_values(["k", "r"]).index.to_numpy()
        batches = [order[i:i + E5_BATCH] for i in range(0, len(order), E5_BATCH)]
        batches = [batches[i] for i in rng.permutation(len(batches))]
        enc = par(Pooled(e5))

        def e5_loss(b):   # symmetric InfoNCE = S1->S2/S3 and S2/S3->S1 selection
            za = enc(**toks(e5_tok, ["query: " + t for t in take(tr.txt, a_si[b])]))
            zb = enc(**toks(e5_tok, ["query: " + t for t in take(tr.txt, b_qi[b])]))
            lg = (za.float() @ zb.float().T) * 20.0
            ids = torch.as_tensor(a_si[b], device=DEV)
            same = (ids[:, None] == ids[None, :]) & ~torch.eye(len(b), dtype=torch.bool, device=DEV)
            lg = lg.masked_fill(same, -1e4)   # other true matches of the same S1 are not negatives
            lab = torch.arange(len(b), device=DEV)
            return (F.cross_entropy(lg, lab) + F.cross_entropy(lg.T, lab)) / 2

        log("E5-TRAIN", f"{len(sel):,} positive pairs, {len(batches):,} batches")
        train_loop("E5-TRAIN", enc, batches, e5_loss, E5_LR, E5_MAX_MIN)
        torch.save(e5.state_dict(), ck("e5.pt"))
        del enc, a_si, b_qi, order, batches
DIM = e5.config.hidden_size

In [ ]:
# Stage 3: encode every record (both GPUs), memmapped fp16
@torch.no_grad()
def encode(rec, path, bs=1024):
    if not os.path.exists(path):
        enc = par(Pooled(e5)).eval()
        n = len(rec); tmp = path + ".tmp.npy"
        out = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, DIM))
        order = np.argsort(rec.txt.str.len().to_numpy(), kind="stable")
        t = time.time()
        for i in range(0, n, bs):
            ix = order[i:i + bs]
            with torch.autocast(**AMP):
                out[ix] = enc(**toks(e5_tok, ["query: " + s for s in take(rec.txt, ix)])).float().cpu().numpy()
            if (i // bs) % 1000 == 0:
                el = time.time() - t
                log("ENCODE", f"{i + len(ix):,}/{n:,} | {(i + len(ix)) / el:,.0f} rec/s | ETA {el / (i + len(ix)) * (n - i) / 60:.0f} min")
        out.flush(); del out, enc; os.replace(tmp, path)
    return np.load(path, mmap_mode="r")

In [ ]:
# Stage 4 (v4): helpers
def h64(parts):
    s = parts[0].astype(str)
    for p in parts[1:]:
        s = s + "|" + p.astype(str)
    return pd.util.hash_array(s.to_numpy(dtype=object))


def name_num_keys(rec):   # same country + same name + shared address number
    base = [rec.cc, rec.nk]
    return [np.where((rec[f"n{j}"] > 0) & (rec.nk.str.len() > 0), h64(base + [rec[f"n{j}"]]), 0) for j in range(3)]


def addr_keys(rec):       # same country + same address token multiset (S3 reorders components)
    srt = rec.ak.astype(object).map(lambda s: " ".join(sorted(s.split())))
    return [np.where(srt.str.len() > 5, h64([rec.cc, srt]), 0)]


def grp(g, v):
    """Per-row rank (by v desc) inside group g, plus group max, second max and size, all per row.
    Low-memory version: int32 indices, v's dtype for values, temporaries freed as soon as possible."""
    n = len(g)
    o = np.lexsort((-v, g))
    gs = g[o]
    newg = np.empty(n, bool); newg[:1] = True
    if n > 1:
        np.not_equal(gs[1:], gs[:-1], out=newg[1:])
    del gs
    start = np.flatnonzero(newg)
    gid = np.cumsum(newg, dtype=np.int32); gid -= 1; del newg
    size = np.diff(np.r_[start, n]).astype(np.int32)
    vs = v[o]
    best_g = vs[start]
    sec_g = np.where(size > 1, vs[np.minimum(start + 1, n - 1)], -1).astype(v.dtype)
    del vs
    rank = np.empty(n, np.int32); rank[o] = np.arange(n, dtype=np.int32) - start.astype(np.int32)[gid]
    best = np.empty(n, v.dtype); best[o] = best_g[gid]
    sec = np.empty(n, v.dtype); sec[o] = sec_g[gid]
    sz = np.empty(n, np.int32); sz[o] = size[gid]
    del o, gid
    return [rank, best, sec, sz]


def pair_key(si, qi):
    return si.astype(np.int64) * (1 << 32) + qi.astype(np.int64)


def macro_f(si_pred, y_pred, n_true, mask):
    """Exact challenge metric: F0.5 per S1 (singleton = 1 iff empty), averaged over S1 rows in mask."""
    npred = np.bincount(si_pred, minlength=len(n_true))
    tp = np.bincount(si_pred, weights=y_pred, minlength=len(n_true))
    with np.errstate(divide="ignore", invalid="ignore"):
        P, R = tp / npred, tp / n_true
        f = np.where(n_true == 0, (npred == 0).astype(float), np.where(tp > 0, 1.25 * P * R / (0.25 * P + R), 0.0))
    return float(f[mask].mean()), f


def decode(si, qi, p, uniq, rule, t):
    keep = p >= P_FLOOR
    if uniq:   # each S2/S3 record goes to its single best S1 (training prior: 0 multi-owner records)
        keep &= grp(qi, p + 1e-9 * np.arange(len(p)) / len(p))[0] == 0
    ix = np.flatnonzero(keep)
    if rule == "thr":
        return ix[p[ix] >= t]
    pc = np.clip(p[ix], 1e-6, 1 - 1e-6) ** t      # t = calibration exponent
    o = np.lexsort((-pc, si[ix])); s, q_ = si[ix][o], pc[o]
    start = np.r_[0, np.flatnonzero(s[1:] != s[:-1]) + 1]
    size = np.diff(np.r_[start, len(s)])
    gid = np.repeat(np.arange(len(start)), size)
    k = np.arange(len(s)) - start[gid] + 1
    cs = np.cumsum(q_); cs -= np.r_[0, cs[start[1:] - 1]][gid]
    N = np.add.reduceat(q_, start)[gid]
    ef = 1.25 * cs / (0.25 * N + k)               # plug-in E[F0.5] of predicting the top-k
    e0 = np.exp(np.add.reduceat(np.log1p(-q_), start))   # P(no true match) = E[F0.5] of predicting {}
    best = np.maximum.reduceat(ef, start)
    kbest = np.minimum.reduceat(np.where(ef == best[gid], k, len(s) + 1), start)
    kstar = np.where(best > e0, kbest, 0)
    return ix[o][k <= kstar[gid]]


def tune(v, ws):
    rows = []
    si, qi, y = v.si.to_numpy(), v.qi.to_numpy(), v.y.to_numpy()
    for w in ws:
        p = v.p1.to_numpy() if w == 0 else np.where(v.routed, w * v.pce + (1 - w) * v.p1, v.p1)
        for uniq in (0, 1):
            for rule, grid in (("thr", np.arange(0.1, 0.96, 0.05)), ("efb", (0.5, 0.75, 1, 1.5, 2, 3))):
                for t in grid:
                    sel = decode(si, qi, p, uniq, rule, t)
                    rows.append(dict(w=w, uniq=uniq, rule=rule, t=round(float(t), 3), f05=macro_f(si[sel], y[sel], n_true, vmask)[0]))
    sw = pd.DataFrame(rows).sort_values("f05", ascending=False)
    return sw, sw.iloc[0].to_dict()


def write_lists(rec, n1, si, qi, path, col):
    ids = rec.entity_id.astype(object).to_numpy()
    o = np.argsort(si, kind="stable"); si, qi = np.asarray(si)[o], np.asarray(qi)[o]
    bounds = np.searchsorted(si, np.arange(n1 + 1))   # rows of S1 i are bounds[i]:bounds[i+1]
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for i in range(n1):
            f.write(f"{ids[i]}\t{','.join(ids[qi[bounds[i]:bounds[i + 1]]])}\n")
    os.replace(path + ".tmp", path)


def write_submission(te, n1, tc, p, cfg, note):
    sel = decode_final(tc.si.to_numpy(), tc.qi.to_numpy(), p, cfg["uniq"], cfg["rule"], cfg["t"], DKEY_TE)
    write_lists(te, n1, tc.si.to_numpy()[sel], tc.qi.to_numpy()[sel], f"{OUT}/matching_results.tsv", "matched_entity_ids")
    if not os.path.exists(f"{OUT}/candidate_pairs.tsv"):
        write_lists(te, n1, tc.si.to_numpy(), tc.qi.to_numpy(), f"{OUT}/candidate_pairs.tsv", "candidate_entity_ids")
    npred = np.bincount(tc.si.to_numpy()[sel], minlength=n1)
    M[f"test_{note}"] = dict(links=len(sel), singleton_rate=float((npred == 0).mean()), cfg=cfg)
    log("SUBMIT", f"[{note}] {len(sel):,} links | predicted singletons {(npred == 0).mean():.3%} | cfg {cfg}")
    if VALIDATOR:
        r = subprocess.run([sys.executable, VALIDATOR, "--matching", f"{OUT}/matching_results.tsv",
                            "--candidate", f"{OUT}/_skip.tsv",   # quick in-run check; full check runs at the end
                            "--test-dir", os.path.dirname(TSVS["test_source1.tsv"])],
                           capture_output=True, text=True)
        log("VALIDATE", (r.stdout + r.stderr)[-1500:])
    dump()


def amb(p):   # distance from the decision boundary in logit space (small = ambiguous)
    return np.abs(np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1)))


@torch.no_grad()
def ce_score(model, tok, rec, si, qi, shard=None, bs=256):
    out = np.zeros(len(si), np.float32)
    order = np.argsort(rec.txt.str.len().to_numpy()[si] + rec.txt.str.len().to_numpy()[qi], kind="stable")
    step = 500_000; t = time.time()
    if shard:   # shard names carry a fingerprint of the exact pair list, so another run's chunks can never be reused
        import hashlib
        shard = f"{shard}_{hashlib.md5(np.ascontiguousarray(pair_key(np.asarray(si), np.asarray(qi))).tobytes()).hexdigest()[:12]}"
    for j in range(0, len(si), step):
        f = ck(f"{shard}_{j // step}.npy") if shard else None   # finds shards of an attached previous run
        ix = order[j:j + step]
        if f and os.path.exists(f):
            prev = np.load(f)
            if len(prev) == len(ix):
                out[ix] = prev; log("CE", f"reused shard {os.path.basename(f)}"); continue
            log("CE", f"shard {os.path.basename(f)} has the wrong length ({len(prev):,} vs {len(ix):,}); rescoring")
        res = np.empty(len(ix), np.float32)
        for b in range(0, len(ix), bs):
            bb = ix[b:b + bs]
            with torch.autocast(**AMP):
                res[b:b + bs] = torch.sigmoid(model(**toks(tok, take(rec.txt, si[bb]), take(rec.txt, qi[bb]), MAXLEN_CE)).float()).cpu().numpy()
        out[ix] = res
        if f:
            np.save(f, res)
        log("CE", f"{min(j + step, len(si)):,}/{len(si):,} pairs | {min(j + step, len(si)) / (time.time() - t):,.0f} pairs/s")
    return out, len(si) / max(time.time() - t, 1e-6)


@torch.no_grad()
def encode_texts(texts, path, bs=1024, tag="ENCODE"):
    """E5 embeddings for an arbitrary list of texts -> fp16 memmap aligned with `texts`."""
    if not os.path.exists(path):
        enc = par(Pooled(e5)).eval()
        n = len(texts); tmp = path + ".tmp.npy"
        out = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(max(n, 1), DIM))
        order = np.argsort(np.fromiter((len(s) for s in texts), np.int64, n), kind="stable")
        t = time.time()
        for i in range(0, n, bs):
            ix = order[i:i + bs]
            with torch.autocast(**AMP):
                out[ix] = enc(**toks(e5_tok, ["query: " + texts[j] for j in ix])).float().cpu().numpy()
            if (i // bs) % 1000 == 0:
                el = time.time() - t
                log(tag, f"{i + len(ix):,}/{n:,} | {(i + len(ix)) / el:,.0f} rec/s | ETA {el / (i + len(ix)) * (n - i) / 60:.0f} min")
        out.flush(); del out, enc; os.replace(tmp, path)
    return np.load(path, mmap_mode="r")


def names(rec, rows):
    return np.array([s.split(" ; ", 1)[0] for s in take(rec.txt, rows)], dtype=object)


def dense_topk(Es, s_pos, s_rows, Eq, q_pos, q_rows, cc, k, tag):
    """Per country: each query row -> top-k S1 rows by cosine. Es[s_pos[i]] embeds s_rows[i]; same for q."""
    DT = torch.float16 if DEV == "cuda" else torch.float32
    dq, ds = [], []
    for c in (np.unique(cc[q_rows]) if PER_COUNTRY else [None]):
        sm = cc[s_rows] == c if c is not None else np.ones(len(s_rows), bool)
        qm = cc[q_rows] == c if c is not None else np.ones(len(q_rows), bool)
        if not sm.any() or not qm.any():
            continue
        S = torch.from_numpy(np.ascontiguousarray(Es[s_pos[sm]])).to(DEV, DT)
        sr, qp, qr = s_rows[sm], q_pos[qm], q_rows[qm]
        kk = min(k, len(sr)); qb = min(65536, max(256, int(4e9 / 2 / len(sr))))
        for j in range(0, len(qr), qb):
            i = (torch.from_numpy(np.ascontiguousarray(Eq[qp[j:j + qb]])).to(DEV, DT) @ S.T).topk(kk, dim=1).indices
            dq.append(np.repeat(qr[j:j + qb], kk)); ds.append(sr[i.cpu().numpy().ravel()])
        log(tag, f"country {c}: {len(sr):,} S1 x {len(qr):,} queries")
        del S
    return pair_key(np.concatenate(ds), np.concatenate(dq)) if dq else np.zeros(0, np.int64)


def exact_name_keys(rec):
    """Name-only exact keys for the empty-address rescue: core name, name without spaces, sorted name tokens."""
    nk = rec.nk.astype(object)
    out = []
    for ser in (pd.Series(core_names(rec)), nk.map(lambda s: s.replace(" ", "")), nk.map(lambda s: " ".join(sorted(s.split())))):
        out.append(np.where(ser.str.len().to_numpy() > 3, h64([rec.cc, ser]), 0))
    return out


def augment_exact(c, rec, n1, s_ok, q_ok, path, cos_fn, cap=30):
    """Adds pairs for EMPTY-ADDRESS records whose name matches an S1 name exactly (in a normalised form), where at
    most `cap` S1s share that name. Existing pairs keep their scores; group features are recomputed."""
    if os.path.exists(ck(path)):
        return pd.read_parquet(ck(path))
    with stage(f"RESCUE-{path}"):
        qe = q_ok & (rec.ak.str.len().to_numpy() == 0)
        ke = key_join_m(rec, n1, exact_name_keys, cap, s_ok, qe)
        # plus a wider name-only E5 search (top-KN_RESCUE) for empty-address records: catches typos / partial names
        km = name_channel(rec, n1, s_ok, qe, "rescue_" + path.split(".")[0], k=KN_RESCUE)
        log("RESCUE", f"exact-name pairs {len(ke):,} | name-only top-{KN_RESCUE} pairs {len(km):,}")
        ke = np.unique(np.concatenate([ke, km]))
        base = c[["qi", "si", "d", "n", "a", "m", "cos"]]
        k_old = pair_key(base.si.values, base.qi.values)
        new = ke[~np.isin(ke, k_old)]
        nsi, nqi = (new >> 32).astype(np.int64), (new & 0xFFFFFFFF).astype(np.int64)
        cos_new = cos_fn(nsi, nqi) if len(new) else np.zeros(0, np.float32)
        keys = np.concatenate([k_old, new]); o = np.argsort(keys, kind="stable"); keys = keys[o]
        z = np.zeros(len(new), np.int8)
        flags = {x: np.concatenate([base[x].to_numpy(), z])[o] for x in ("d", "n", "a")}
        flags["m"] = (np.concatenate([base.m.to_numpy(), np.ones(len(new), np.int8)])[o] | np.isin(keys, ke)).astype(np.int8)
        out = frame_from_keys(keys, flags)
        out["cos"] = np.concatenate([base.cos.to_numpy(), cos_new])[o]
        del base, k_old, keys, o, z, flags; trim()
        out = add_group_feats(out)
        log("RESCUE", f"{int(qe.sum()):,} empty-address records | {len(ke):,} rescue pairs (exact + name top-K) | {len(new):,} new | {len(out):,} total | {mem()}")
        out.to_parquet(f"{CKPT}/{path}")
    return out


def key_join_m(rec, n1, keys_fn, cap, s_ok, q_ok):
    out = []
    for k in keys_fn(rec):
        valid = k != 0
        sv = np.flatnonzero(valid[:n1] & s_ok[:n1])
        s = pd.DataFrame({"k": k[sv], "si": sv})
        vc = s.k.value_counts()
        s = s[s.k.map(vc).to_numpy() <= cap]
        qv = np.flatnonzero(valid[n1:] & q_ok[n1:]) + n1
        out.append(pd.DataFrame({"k": k[qv], "qi": qv}).merge(s, on="k")[["qi", "si"]])
    ex = pd.concat(out, ignore_index=True)
    return np.unique(pair_key(ex.si.values, ex.qi.values))


def cos_pairs(E, si, qi, pos=None):
    out = np.empty(len(si), np.float32)
    for j in range(0, len(si), 1_000_000):   # 1M-row chunks: ~1.5 GB transient instead of ~3 GB
        a_i, b_i = si[j:j + 1_000_000], qi[j:j + 1_000_000]
        if pos is not None:
            a_i, b_i = np.searchsorted(pos, a_i), np.searchsorted(pos, b_i)
        a = torch.from_numpy(np.asarray(E[a_i])).to(DEV).float()
        b = torch.from_numpy(np.asarray(E[b_i])).to(DEV).float()
        out[j:j + 1_000_000] = (a * b).sum(1).cpu().numpy()
    return out


def add_group_feats(c):
    cos = c.cos.to_numpy()
    r, best, sec, size = grp(c.qi.values, cos)
    c["rank_q"], c["gap_q"], c["marg_q"], c["n_q"] = r.astype(np.int16), best - cos, best - np.maximum(sec, -1), size.astype(np.int16)
    r, best, _, size = grp(c.si.values, cos)
    c["rank_s"], c["gap_s"], c["n_s"] = r.astype(np.int16), best - cos, size.astype(np.int32)
    return c


def frame_from_keys(keys, flags):
    c = pd.DataFrame({"qi": (keys & 0xFFFFFFFF).astype(np.int32), "si": (keys >> 32).astype(np.int32)})
    for name, v in flags.items():
        c[name] = v.astype(np.int8)
    return c


def name_channel(rec, n1, s_ok, q_ok, tag, k=None):
    """Empty-address S2/S3 records -> top-KN S1 by NAME-ONLY embedding (the full-text one is diluted by S1 addresses)."""
    cc = rec.cc.to_numpy()
    s_rows = np.flatnonzero(s_ok[:n1])
    q_rows = np.flatnonzero(q_ok[n1:] & (rec.ak.str.len().to_numpy()[n1:] == 0)) + n1
    log(tag, f"name channel: {len(q_rows):,} empty-address records vs {len(s_rows):,} S1 names")
    NS = encode_texts(names(rec, s_rows), ck(f"nameemb_{tag}_s1.npy", big=True), tag="ENCODE-NAMES")
    NQ = encode_texts(names(rec, q_rows), ck(f"nameemb_{tag}_q.npy", big=True), tag="ENCODE-NAMES")
    return dense_topk(NS, np.arange(len(s_rows)), s_rows, NQ, np.arange(len(q_rows)), q_rows, cc, k or KN, tag)


NOISE_TOK = set("""llc inc corp corporation co company ltd limited pvt private llp lp pc pllc plc incorporated
    sarl sas sasu eurl sa sci snc ei the and center centre services service group enterprises enterprise trading
    holdings com www dba formerly fka aka doing business as
    fils associes cie compagnie developpement participations et""".split())


def core_names(rec):   # name without legal forms / injected filler words (both sides get them at random)
    return rec.nk.astype(object).map(lambda s: " ".join(t for t in s.split() if t not in NOISE_TOK)).to_numpy(dtype=object)


def num_matrix(rec):   # up to 6 distinct address numbers, sorted, 0 = empty (S3 reorders, so compare as sets)
    M_ = np.zeros((len(rec), 6), np.int64)
    for i, l in enumerate(rec.ak.astype(object).map(lambda s: sorted({int(t[-15:]) + 1 for t in re.findall(r"\d+", s)})[:6])):
        if l:
            M_[i, :len(l)] = l
    return M_


def freq_arrays(rec, n1, s_ok):
    """How many S1s (within the country) share this record's name / first-2-token name / address token set."""
    tok2 = rec.nk.astype(object).map(lambda s: " ".join(s.split()[:2]))
    srt = rec.ak.astype(object).map(lambda s: " ".join(sorted(s.split())))
    keys = {"name": (h64([rec.cc, rec.nk]), rec.nk.str.len().to_numpy() > 0),
            "name2": (h64([rec.cc, tok2]), tok2.str.len().to_numpy() > 0),
            "addr": (h64([rec.cc, srt]), srt.str.len().to_numpy() > 5)}
    sm = np.zeros(len(rec), bool); sm[:n1] = s_ok[:n1]
    out = {"num6": num_matrix(rec), "core": core_names(rec)}
    for name, (k, valid) in keys.items():
        vc = pd.Series(k[sm & valid]).value_counts()
        f = pd.Series(k).map(vc).fillna(0).to_numpy(np.float32)
        f[~valid] = -1
        out[name] = f
    return out


FEATS4 = ["cos", "rank_q", "gap_q", "marg_q", "n_q", "rank_s", "gap_s", "n_s", "d", "n", "a", "m",
          "name_ratio", "name_tset", "addr_tsort", "addr_tset", "num_match", "num_q", "num_s", "num0_eq",
          "ak_empty_q", "ak_empty_s", "is_s3", "len_ratio",
          "name_freq_s", "name_freq_q", "name2_freq_q", "addr_freq_s", "addr_freq_q",
          "num0_lev", "num0_prefix", "num0_absdiff",
          "nset_eq", "nset_jacc", "nset_disjoint", "core_tset", "core_ratio",
          "tf_name", "tf_core", "tf_addr", "tf_rank_q", "tf_gap_q", "tf_rank_s", "tf_gap_s", "fcos", "frank_q", "fgap_q"]


def pair_feats4(c, rec, FQ):
    si, qi = c.si.values, c.qi.values
    X = {f: c[f].to_numpy(np.float32) for f in FEATS4[:12] + ["tf_name", "tf_core", "tf_addr", "tf_rank_q", "tf_gap_q", "tf_rank_s", "tf_gap_s", "fcos", "frank_q", "fgap_q"]}
    n1, n2 = take(rec.nk, si), take(rec.nk, qi)
    a1, a2 = take(rec.ak, si), take(rec.ak, qi)
    X["name_ratio"] = cpdist(n1, n2, scorer=fuzz.ratio, workers=-1)
    X["name_tset"] = cpdist(n1, n2, scorer=fuzz.token_set_ratio, workers=-1)
    X["addr_tsort"] = cpdist(a1, a2, scorer=fuzz.token_sort_ratio, workers=-1)
    X["addr_tset"] = cpdist(a1, a2, scorer=fuzz.token_set_ratio, workers=-1)
    NS = rec[["n0", "n1", "n2"]].to_numpy()
    ns, nq = NS[si], NS[qi]
    X["num_match"] = ((nq[:, :, None] == ns[:, None, :]) & (nq[:, :, None] > 0)).any(2).sum(1)
    X["num_q"], X["num_s"] = (nq > 0).sum(1), (ns > 0).sum(1)
    X["num0_eq"] = np.where((nq[:, 0] > 0) & (ns[:, 0] > 0), (nq[:, 0] == ns[:, 0]).astype(np.int8) * 2 - 1, 0)
    L = rec.ak.str.len().to_numpy()
    X["ak_empty_q"], X["ak_empty_s"] = L[qi] == 0, L[si] == 0
    X["is_s3"] = rec.src.to_numpy()[qi] == 3
    ln = rec.nk.str.len().to_numpy()
    X["len_ratio"] = np.minimum(ln[si], ln[qi]) / np.maximum(np.maximum(ln[si], ln[qi]), 1)
    X["name_freq_s"], X["name_freq_q"], X["name2_freq_q"] = FQ["name"][si], FQ["name"][qi], FQ["name2"][qi]
    X["addr_freq_s"], X["addr_freq_q"] = FQ["addr"][si], FQ["addr"][qi]
    a0, b0 = nq[:, 0], ns[:, 0]
    both = (a0 > 0) & (b0 > 0)
    sa = np.where(both, (a0 - 1).astype(str), ""); sb = np.where(both, (b0 - 1).astype(str), "")
    X["num0_lev"] = np.where(both, cpdist(sa.tolist(), sb.tolist(), scorer=Levenshtein.distance, workers=-1), -1)
    X["num0_prefix"] = np.where(both, (np.char.startswith(sa, sb) | np.char.startswith(sb, sa)).astype(np.float32), -1)
    X["num0_absdiff"] = np.where(both, np.log1p(np.abs(a0 - b0).astype(np.float64)), -1)
    A, B = FQ["num6"][qi], FQ["num6"][si]   # full number sets: chain siblings almost never share them, true pairs usually do
    na, nb = (A > 0).sum(1), (B > 0).sum(1)
    inter = ((A[:, :, None] == B[:, None, :]) & (A[:, :, None] > 0)).any(2).sum(1)
    hasb = (na > 0) & (nb > 0)
    X["nset_eq"] = np.where(hasb, (A == B).all(1), -1)
    X["nset_jacc"] = np.where(hasb, inter / np.maximum(na + nb - inter, 1), -1)
    X["nset_disjoint"] = np.where(hasb, inter == 0, -1)
    c1, c2 = FQ["core"][si], FQ["core"][qi]
    X["core_tset"] = cpdist(c1, c2, scorer=fuzz.token_set_ratio, workers=-1)
    X["core_ratio"] = cpdist(c1, c2, scorer=fuzz.ratio, workers=-1)
    return np.column_stack([np.asarray(X[f], np.float32) for f in FEATS4])


def feats4(c, rec, FQ, fn, step=4_000_000):
    out = []
    for j in range(0, len(c), step):
        ch = c.iloc[j:j + step]
        out.append(fn(pair_feats4(ch, rec, FQ), ch))
        log("FEATS", f"{min(j + step, len(c)):,}/{len(c):,}")
    return np.concatenate(out)


DUP = False   # duplicate consistency, switched on below only if validation improves


def dup_keys(rec, n1):
    """Group id for S2/S3 records with identical normalised text (0 = no duplicate)."""
    k = np.zeros(len(rec), np.int64)
    h = pd.util.hash_array(rec.txt.astype(object).to_numpy()[n1:]).astype(np.int64)
    cnt = pd.Series(h).map(pd.Series(h).value_counts()).to_numpy()
    k[n1:] = np.where(cnt >= 2, h, 0)
    return k


def dup_fill(si, qi, p, sel, dkey):
    """If every assigned copy of a duplicate group went to one S1, give the unassigned copies to it too
    (only where that pair is a candidate). Train: identical records share one S1 96% of the time."""
    q_sel, s_sel = qi[sel], si[sel]
    g = dkey[q_sel]
    df = pd.DataFrame({"g": g, "s": s_sel})[g != 0]
    if not len(df):
        return sel
    nun = df.groupby("g").s.nunique()
    one = nun[nun == 1].index.to_numpy()
    owner_of = df[df.g.isin(one)].drop_duplicates("g").set_index("g").s
    cg = dkey[qi]
    ix = np.flatnonzero(np.isin(cg, one) & (p >= P_FLOOR))
    add = ix[si[ix] == owner_of.reindex(cg[ix]).to_numpy()]
    add = add[~np.isin(qi[add], q_sel)]
    return np.union1d(sel, add)


def decode_final(si, qi, p, uniq, rule, t, dkey):
    sel = decode(si, qi, p, uniq, rule, t)
    return dup_fill(si, qi, p, sel, dkey) if DUP else sel


def num_conflict(si, qi, FQ, step=4_000_000):
    """Both addresses have numbers and share none (the brief's 'neighbouring house number' hard negatives)."""
    out = np.zeros(len(si), bool)
    for j in range(0, len(si), step):
        A, B = FQ["num6"][qi[j:j + step]], FQ["num6"][si[j:j + step]]
        inter = ((A[:, :, None] == B[:, None, :]) & (A[:, :, None] > 0)).any(2).sum(1)
        out[j:j + step] = ((A > 0).any(1)) & ((B > 0).any(1)) & (inter == 0)
    return out


def blend(p1, pce, routed, conf, w, veto):
    """Final score. veto=1: the reranker may not overrule an XGB 'no' when the house numbers conflict."""
    p = np.where(routed, w * pce + (1 - w) * p1, p1)
    if veto:
        p = np.where(routed & conf & (p1 < 0.5), p1, p)
    return p.astype(np.float32)


def tune2(v, ws, vetos=(0, 1)):
    rows = []
    si, qi, y = v.si.to_numpy(), v.qi.to_numpy(), v.y.to_numpy()
    for w in ws:
        for veto in (vetos if w > 0 else (0,)):
            p = v.p1.to_numpy() if w == 0 else blend(v.p1.to_numpy(), v.pce.to_numpy(), v.routed.to_numpy(), v.conf.to_numpy(), w, veto)
            for uniq in (0, 1):
                for rule, grid in (("thr", np.arange(0.1, 0.96, 0.05)), ("efb", (0.5, 0.75, 1, 1.5, 2, 3))):
                    for t in grid:
                        sel = decode(si, qi, p, uniq, rule, t)
                        rows.append(dict(w=w, veto=veto, uniq=uniq, rule=rule, t=round(float(t), 3),
                                         f05=macro_f(si[sel], y[sel], n_true, vmask)[0]))
    sw = pd.DataFrame(rows).sort_values("f05", ascending=False)
    return sw, sw.iloc[0].to_dict()


def tune_fast2(v, ws):
    keep = (v.p1.to_numpy() >= P_FLOOR) | (v.routed.to_numpy() & (v.pce.to_numpy() >= P_FLOOR))
    return tune2(v[keep].reset_index(drop=True), ws)


def tune_fast(v, ws):   # rows that can never reach P_FLOOR cannot change any decision
    keep = v.p1.to_numpy() >= P_FLOOR
    if "routed" in v:
        keep |= v.routed.to_numpy() & (v.pce.to_numpy() >= P_FLOOR)
    return tune(v[keep].reset_index(drop=True), ws)

In [ ]:
# Stage 4 (v4.2): TF-IDF similarity (rare tokens count most) on names, core names and normalised addresses
US_STATES = {"alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca", "colorado": "co",
             "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga", "hawaii": "hi", "idaho": "id",
             "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
             "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi", "minnesota": "mn",
             "mississippi": "ms", "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
             "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny", "north carolina": "nc",
             "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa",
             "rhode island": "ri", "south carolina": "sc", "south dakota": "sd", "tennessee": "tn", "texas": "tx",
             "utah": "ut", "vermont": "vt", "virginia": "va", "washington": "wa", "west virginia": "wv",
             "wisconsin": "wi", "wyoming": "wy", "district of columbia": "dc"}
IN_STATES = {"andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as", "bihar": "br", "chhattisgarh": "cg",
             "goa": "ga", "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp", "jharkhand": "jh",
             "karnataka": "ka", "kerala": "kl", "keralam": "kl", "madhya pradesh": "mp", "maharashtra": "mh",
             "manipur": "mn", "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl", "odisha": "od", "orissa": "od",
             "punjab": "pb", "rajasthan": "rj", "sikkim": "sk", "tamil nadu": "tn", "telangana": "tg", "ts": "tg",
             "tripura": "tr", "uttar pradesh": "up", "uttarakhand": "uk", "west bengal": "wb", "delhi": "dl"}
IN_NATIVE = {"महाराष्ट्र": "mh", "दिल्ली": "dl", "उत्तर प्रदेश": "up", "मध्य प्रदेश": "mp", "ಕರ್ನಾಟಕ": "ka",
             "தமிழ்நாடு": "tn", "తెలంగాణ": "tg", "পশ্চিমবঙ্গ": "wb", "ગુજરાત": "gj", "हरियाणा": "hr",
             "राजस्थान": "rj", "കേരളം": "kl", "बिहार": "br", "ఆంధ్రప్రదేశ్": "ap", "ਪੰਜਾਬ": "pb", "ଓଡ଼ିଶା": "od",
             "झारखंड": "jh", "उत्तराखंड": "uk", "छत्तीसगढ़": "cg", "গোয়া": "ga", "गोवा": "ga", "অসম": "as"}
FR_REGIONS = ["nouvelle aquitaine", "hauts de france", "pays de la loire"]
STREET_ABBR = {"r": "rue", "bd": "blvd", "boulevard": "blvd", "crs": "cours", "ch": "chemin", "chem": "chemin",
               "imp": "impasse", "all": "allee", "rte": "route", "pl": "place", "sq": "square", "fbg": "faubourg",
               "av": "ave", "avenue": "ave", "street": "st", "saint": "st", "road": "rd", "drive": "dr", "lane": "ln",
               "court": "ct", "circle": "cir", "highway": "hwy", "parkway": "pkwy", "terrace": "ter", "trail": "trl",
               "north": "n", "south": "s", "east": "e", "west": "w", "mount": "mt", "fort": "ft"}
ADDR_DROP = {"apt", "apartment", "unit", "ste", "suite", "fl", "floor", "bldg", "building", "rm", "room", "no", "nr",
             "num", "number", "po", "box", "pmb", "cdp", "the", "of", "de", "la", "du", "des", "le", "les", "d", "l",
             "france", "null", "na"}


def _addr_maps():
    phr, single = {}, dict(STREET_ABBR)
    for full, code in {**US_STATES, **IN_STATES}.items():
        (phr if " " in full else single)[full] = code
    for nat, code in IN_NATIVE.items():
        k = keyv(nat)
        if k:
            (phr if " " in k else single)[k] = code
    for reg in FR_REGIONS:
        phr[reg] = ""
    rx = re.compile(r"(?<![a-z0-9#])(" + "|".join(sorted(map(re.escape, phr), key=len, reverse=True)) + r")(?![a-z0-9#])")
    return phr, single, rx


ADDR_PHR, ADDR_SINGLE, ADDR_RX = _addr_maps()


def norm_addr(s):
    s = ADDR_RX.sub(lambda m: ADDR_PHR[m.group(1)], s)
    return " ".join(ADDR_SINGLE.get(t, t) for t in s.split() if t not in ADDR_DROP)


HV = HashingVectorizer(n_features=2 ** 21, alternate_sign=False, norm=None, lowercase=False,
                       token_pattern=r"[^ ]+", dtype=np.float32)


def tfidf_matrix(texts):
    X = HV.transform(texts).tocsr(); X.sort_indices()
    df = np.bincount(X.indices, minlength=X.shape[1])
    X.data = X.data * (np.log((X.shape[0] + 1) / (df + 1)) + 1).astype(np.float32)[X.indices]
    return sk_normalize(X, norm="l2", copy=False)


def rowdot(X, si, qi, step=2_000_000):
    out = np.empty(len(si), np.float32)
    for j in range(0, len(si), step):
        out[j:j + step] = np.asarray(X[si[j:j + step]].multiply(X[qi[j:j + step]]).sum(axis=1)).ravel()
    return out


TF_COLS = ["tf_name", "tf_core", "tf_addr", "tf_rank_q", "tf_gap_q", "tf_rank_s", "tf_gap_s"]

In [ ]:
# Stage 4b (v4.9): similarity from the ORIGINAL multilingual-E5 (never fine-tuned on US/India)
# Our fine-tuned E5 may have lost some of the multilingual geometry France needs. The frozen model's cosine, its
# rank among a record's candidates and its gap to the best are added as features; leave-country-out decides.
FZ_COLS = ["fcos", "frank_q", "fgap_q"]
_e5_frozen = None


@torch.no_grad()
def encode_frozen(rec, path, bs=1024):
    global _e5_frozen
    if os.path.exists(path):
        return np.load(path, mmap_mode="r")
    if _e5_frozen is None:
        _e5_frozen = AutoModel.from_pretrained(E5_MODEL)
    enc = par(Pooled(_e5_frozen)).eval()
    n = len(rec); tmp = path + ".tmp.npy"
    out = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, DIM))
    order = np.argsort(rec.txt.str.len().to_numpy(), kind="stable"); t = time.time()
    for i in range(0, n, bs):
        ix = order[i:i + bs]
        with torch.autocast(**AMP):
            out[ix] = enc(**toks(e5_tok, ["query: " + s_ for s_ in take(rec.txt, ix)])).float().cpu().numpy()
        if (i // bs) % 1000 == 0:
            el = time.time() - t
            log("ENCODE-FROZEN", f"{i + len(ix):,}/{n:,} | {(i + len(ix)) / el:,.0f} rec/s | ETA {el / (i + len(ix)) * (n - i) / 60:.0f} min")
    out.flush(); del out, enc; os.replace(tmp, path)
    return np.load(path, mmap_mode="r")


def add_frozen_cols(c, rec, name, emb_name):
    path = ck(name)
    if os.path.exists(path):
        z = np.load(path)
        for f in FZ_COLS:
            c[f] = z[f].astype(np.float16)
        return c
    FE = encode_frozen(rec, ck(emb_name, big=True))
    with stage(f"FROZEN-{name}"):
        fcos = cos_pairs(FE, c.si.values, c.qi.values)
        trim(); log("FROZEN", f"ranking | {mem()}")
        r, best_, _, _ = grp(c.qi.values, fcos)
        out = dict(fcos=fcos.astype(np.float16), frank_q=r.astype(np.float16), fgap_q=(best_ - fcos).astype(np.float16)); del r, best_
        np.savez(path, **out)
        for f in FZ_COLS:
            c[f] = out[f].astype(np.float16)
        log("FROZEN", f"mean frozen cosine {fcos.mean():.3f} | corr with fine-tuned cosine {np.corrcoef(fcos[:2_000_000], c.cos.values[:2_000_000])[0, 1]:.3f}")
        del out, fcos; trim()
    return c



def add_tf_cols(c, rec, name):
    path = ck(name)   # saved checkpoint (float16, ~1.4 GB for test) so a resume never recomputes it
    if os.path.exists(path):
        z = np.load(path)
        for f in TF_COLS:
            c[f] = z[f].astype(np.float16)
        return c
    with stage(f"TFIDF-{name}"):
        si, qi = c.si.to_numpy(), c.qi.to_numpy()
        out = {}
        for col, texts in (("tf_name", lambda: rec.nk.astype(object).to_numpy()),
                           ("tf_core", lambda: core_names(rec)),
                           ("tf_addr", lambda: rec.ak.astype(object).map(norm_addr).to_numpy(dtype=object))):
            X = tfidf_matrix(texts())
            out[col] = rowdot(X, si, qi).astype(np.float16)
            del X; trim()
            log("TFIDF", f"{col}: mean over pairs {out[col].mean():.3f} | {mem()}")
        s_ = out["tf_name"].astype(np.float32) + out["tf_addr"].astype(np.float32)
        trim(); log("TFIDF", f"ranking | {mem()}")
        r, best_, _, _ = grp(qi, s_); out["tf_rank_q"], out["tf_gap_q"] = r.astype(np.float16), (best_ - s_).astype(np.float16); del r, best_; trim()
        r, best_, _, _ = grp(si, s_); out["tf_rank_s"], out["tf_gap_s"] = r.astype(np.float16), (best_ - s_).astype(np.float16); del r, best_, s_; trim()
        np.savez(path, **out)
        for f in TF_COLS:
            c[f] = out[f].astype(np.float16)
        del out; gc.collect()
    return c


def load_key_cache(names):
    ks, ss = [], []
    for nm in names:
        p = ck(nm)
        if os.path.exists(p):
            z = np.load(p); ks.append(z["keys"]); ss.append(z["scores"])
    if not ks:
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    k, i = np.unique(np.concatenate(ks), return_index=True)
    return k, np.concatenate(ss)[i]


def lookup(ck_keys, ck_scores, kr):
    if len(ck_keys):
        pos = np.minimum(np.searchsorted(ck_keys, kr), len(ck_keys) - 1); hit = ck_keys[pos] == kr
    else:
        pos = np.zeros(len(kr), np.int64); hit = np.zeros(len(kr), bool)
    sc = np.zeros(len(kr), np.float32); sc[hit] = ck_scores[pos[hit]]
    return sc, hit


def ce_cached(rec, si, qi, cache_name, shard):
    """Cross-encoder scores with a pair-keyed cache saved to ckpt (reused by any later run)."""
    kr = pair_key(si, qi)
    sc, hit = lookup(*load_key_cache([cache_name]), kr)
    miss = np.flatnonzero(~hit); rate = 2300.0
    if len(miss):
        sc[miss], rate = ce_score(ce_model, ce_tok, rec, si[miss], qi[miss], shard=shard)
    np.savez(f"{CKPT}/{cache_name}", keys=kr, scores=sc)
    log("CE", f"{len(kr):,} pairs | {int(hit.sum()):,} from cache | {len(miss):,} scored")
    return sc, rate

In [ ]:
# Stage 4 (v4): clean training world — no record E5 was trained on, all S1 owners visible
# E5 was fine-tuned on records owned by A1 S1s, which made the old validation too easy. The world
# used for XGBoost + validation is: S1 = A2 ∪ V, S2/S3 = records they own + WORLD_DISTRACTOR_PCT% of
# unowned records (keeps the distractor ratio). Retrieval is redone inside this world.
owner = np.full(len(tr), -1, np.int64); owner[t_qi] = t_si
s_ok_tr = np.zeros(len(tr), bool); s_ok_tr[:N1] = part != 0
qh = pd.util.hash_pandas_object(tr.entity_id[N1:], index=False).to_numpy() % 100
q_ok_tr = np.zeros(len(tr), bool)
q_ok_tr[N1:] = np.where(owner[N1:] >= 0, part[np.maximum(owner[N1:], 0)] != 0, qh < WORLD_DISTRACTOR_PCT)
cc_tr = tr.cc.to_numpy()

if os.path.exists(ck("cand_world_v41.parquet")):
    cw = pd.read_parquet(ck("cand_world_v41.parquet"))
else:
    with stage("WORLD"):
        E = encode(tr, ck("emb_train.npy", big=True))
        s_rows = np.flatnonzero(s_ok_tr[:N1]); q_rows = np.flatnonzero(q_ok_tr[N1:]) + N1
        kd = dense_topk(E, s_rows, s_rows, E, q_rows, q_rows, cc_tr, K, "WORLD")
        kn = key_join_m(tr, N1, name_num_keys, 50, s_ok_tr, q_ok_tr)
        ka = key_join_m(tr, N1, addr_keys, 20, s_ok_tr, q_ok_tr)
        km = name_channel(tr, N1, s_ok_tr, q_ok_tr, "train")
        log("WORLD", f"dense {len(kd):,} | name+number {len(kn):,} | address {len(ka):,} | name-only {len(km):,}")
        keys = np.unique(np.concatenate([kd, kn, ka, km]))
        cw = frame_from_keys(keys, {"d": np.isin(keys, kd), "n": np.isin(keys, kn), "a": np.isin(keys, ka), "m": np.isin(keys, km)})
        del keys, kd, kn, ka, km; gc.collect()
        cw["cos"] = cos_pairs(E, cw.si.values, cw.qi.values)
        cw = add_group_feats(cw)
        cw["y"] = np.isin(pair_key(cw.si.values, cw.qi.values), pair_key(t_si, t_qi)).astype(np.int8)
        cw.to_parquet(ck("cand_world_v41.parquet"))
        log("WORLD", f"{len(cw):,} pairs | {len(cw) / max(q_ok_tr.sum(), 1):.1f} per record | {mem()}")

_E_tr = encode(tr, ck("emb_train.npy", big=True))
cw = augment_exact(cw, tr, N1, s_ok_tr, q_ok_tr, "cand_world_v49.parquet", lambda a_, b_: cos_pairs(_E_tr, a_, b_))
cw["y"] = np.isin(pair_key(cw.si.values, cw.qi.values), pair_key(t_si, t_qi)).astype(np.int8)
del _E_tr
_bk = pd.read_parquet(ck("cand_world_v41.parquet"), columns=["qi", "si"]) if os.path.exists(ck("cand_world_v41.parquet")) else None
cw["r"] = (~np.isin(pair_key(cw.si.values, cw.qi.values), pair_key(_bk.si.values, _bk.qi.values))).astype(np.int8) if _bk is not None else np.int8(0)
del _bk
cw = add_tf_cols(cw, tr, "tf_world_v49.npz")
cw = add_frozen_cols(cw, tr, "fz_world_v49.npz", "emb_frozen_train.npy")

# blocking report on V S1s (all their records are in the world)
vmask = part == 2
tv = vmask[t_si]
hitk = pair_key(cw.si.values[cw.y.values == 1], cw.qi.values[cw.y.values == 1])
found = np.isin(pair_key(t_si[tv], t_qi[tv]), hitk)
cv_hit = cw[(cw.y.values == 1) & vmask[cw.si.values]]
blk = {"recall": found.mean(), "pairs_per_query": len(cw) / max(q_ok_tr.sum(), 1),
       "oracle_macro_f05": macro_f(cv_hit.si.values, np.ones(len(cv_hit)), n_true, vmask)[0]}
for ch in ("d", "n", "a", "m"):
    blk[f"recall_channel_{ch}"] = (cv_hit[ch] == 1).sum() / tv.sum()
q_noaddr = tr.ak.str.len().to_numpy()[t_qi[tv]] == 0
q_cty = tr.country.to_numpy(dtype=object)[t_qi[tv]]
for name, g in [("empty_addr", q_noaddr), ("country", q_cty)]:
    for v in np.unique(g):
        blk[f"recall_{name}={v}"] = found[g == v].mean()
M["blocking_world"] = blk
for k_, v_ in blk.items():
    log("BLOCK", f"{k_:28s} {v_:.5f}")
miss = np.flatnonzero(tv)[~found][:3000]
pd.DataFrame({"s1_id": take(tr.entity_id, t_si[miss]), "s1": take(tr.txt, t_si[miss]),
              "q_id": take(tr.entity_id, t_qi[miss]), "q": take(tr.txt, t_qi[miss])}
             ).to_csv(f"{DIAG}/blocking_failures.tsv", sep="\t", index=False)
dump()

In [ ]:
# Stage 5 (v4): XGBoost with frequency + number features; out-of-fold scores for every world pair
FQ_tr = freq_arrays(tr, N1, s_ok_tr)
XGB_P = dict(tree_method="hist", device="cuda" if DEV == "cuda" else "cpu", max_depth=8, eta=0.1, subsample=0.8,
             colsample_bytree=0.8, min_child_weight=5, objective="binary:logistic", eval_metric="logloss", seed=SEED)
if all(os.path.exists(ck(f)) for f in ("xgb49_full.json", "xgb49_f0.json", "xgb49_f1.json", "world_p_oof_v49.npy")):
    xm = {f: xgb.Booster(model_file=ck(f"xgb49_{f}.json")) for f in ("f0", "f1", "full")}
    p_oof = np.load(ck("world_p_oof_v49.npy"))
else:
    with stage("XGB4"):
        samp = cw.iloc[np.sort(rng.permutation(len(cw))[:XGB_MAX_PAIRS])]
        X = feats4(samp, tr, FQ_tr, lambda X, ch: X)
        y = samp.y.to_numpy(); fold = samp.si.to_numpy() % 2   # S1-disjoint folds
        xm, iters = {}, []
        trim(); log("XGB4", f"features ready | {mem()}")
        for f in (0, 1):   # model f is trained on the other fold and scores fold f
            dtr = xgb.QuantileDMatrix(X[fold != f], y[fold != f])
            dva = xgb.QuantileDMatrix(X[fold == f], y[fold == f], ref=dtr)
            trim(); log("XGB4", f"fold {f} matrices built | {mem()}")
            b = xgb.train(XGB_P, dtr, 2000, evals=[(dva, "val")], early_stopping_rounds=50, verbose_eval=200)
            iters.append(b.best_iteration + 1); xm[f"f{f}"] = b[: b.best_iteration + 1]
            xm[f"f{f}"].save_model(ck(f"xgb49_f{f}.json"))
            del dtr, dva, b; trim()
        dfull = xgb.QuantileDMatrix(X, y); trim()
        xm["full"] = xgb.train(XGB_P, dfull, int(np.mean(iters) * 1.1)); del dfull
        xm["full"].save_model(ck("xgb49_full.json"))
        imp = xm["full"].get_score(importance_type="gain")
        M["xgb_gain"] = {FEATS4[int(k[1:])]: round(v, 1) for k, v in sorted(imp.items(), key=lambda t: -t[1])}
        del X, y, samp; trim()
    with stage("XGB4-OOF"):
        def _oof(X, ch):
            d = xgb.DMatrix(X)
            return np.where(ch.si.to_numpy() % 2 == 0, xm["f0"].predict(d), xm["f1"].predict(d)).astype(np.float32)
        p_oof = feats4(cw, tr, FQ_tr, _oof)
        np.save(ck("world_p_oof_v49.npy"), p_oof)

# validation set = every pair of V S1s + all competing pairs (any S1) of records that could go to a V S1
vS = vmask[cw.si.values]
live_q = np.unique(cw.qi.values[vS & (p_oof >= P_FLOOR)])
vsel = vS | np.isin(cw.qi.values, live_q)
val = cw.loc[vsel, ["qi", "si", "y", "cos", "d", "n", "a", "m", "r"]].reset_index(drop=True)
val["p1"] = p_oof[vsel]

In [ ]:
# Stage 5b (v4.3): leave-country-out check of unseen-country calibration, on REAL labels
# Train XGBoost on one country only, score the other, and compare three thresholds for the held-out country:
# the source country's best, one matched to the source's predicted singleton rate, one matched to its links per S1.
LCO_PAIRS = 5_000_000
LCO_GRID = np.round(np.arange(0.2, 0.9501, 0.025), 3)


def lco_check():
    """Leave-country-out on real labels, run twice: with and without the frozen-E5 features. Thresholds are tuned
    on the source country's validation rows only (model trained on source A2 S1s, rows are disjoint V S1s) and
    frozen before scoring the held-out country. Also logs pair-level precision/recall per record bucket."""
    path = ck("lco_v49.json")
    if os.path.exists(path):
        return json.load(open(path))
    cty_rows = tr.country.to_numpy(dtype=object)
    countries = sorted(set(cty_rows[:N1]))
    if len(countries) < 2:
        return {}
    FZIDX = [FEATS4.index(c_) for c_ in FZ_COLS]

    def nofz(X):
        X = X.copy(); X[:, FZIDX] = np.nan
        return X

    variants = {"full": lambda X: X, "nofrozen": nofz}
    res = {}
    with stage("LCO"):
        cw_c = cty_rows[cw.si.values]; in_a2 = part[cw.si.values] == 1
        models = {}
        for c in countries:
            idx = np.sort(rng.permutation(np.flatnonzero(in_a2 & (cw_c == c)))[:LCO_PAIRS])
            samp = cw.iloc[idx]
            X = feats4(samp, tr, FQ_tr, lambda X, ch: X); yv = samp.y.to_numpy(); del samp
            for vn, vf in variants.items():
                d = xgb.QuantileDMatrix(vf(X), yv); models[(vn, c)] = xgb.train(XGB_P, d, 800); del d; trim()
            del X; trim()
            log("LCO", f"models trained on {c} only (with / without frozen E5) | {mem()}")
        cols = [(vn, c) for vn in variants for c in countries]
        vfull = cw.loc[vsel]
        P = feats4(vfull, tr, FQ_tr, lambda X, ch: np.column_stack(
            [models[(vn, c)].predict(xgb.DMatrix(variants[vn](X))) for vn, c in cols]).astype(np.float32))
        del vfull, models; trim()
        si, qi, y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
        vc = cty_rows[si]
        q_empty = tr.ak.str.len().to_numpy()[qi] == 0
        q_nonascii = ~pd.Series(take(tr.txt, qi)).map(str.isascii).to_numpy()
        buckets = {"empty_addr": q_empty, "non_ascii": q_nonascii & ~q_empty, "normal": ~q_empty & ~q_nonascii}

        def curve(p, pair_mask, s1mask):
            ix = np.flatnonzero(pair_mask & (p >= P_FLOOR))
            s_, q_, pp, yy = si[ix], qi[ix], p[ix], y[ix]
            rows = []
            for t in LCO_GRID:
                sel = decode(s_, q_, pp, 1, "thr", t)
                npred = np.bincount(s_[sel], minlength=N1)
                rows.append((float(t), macro_f(s_[sel], yy[sel], n_true, s1mask)[0],
                             float((npred[s1mask] == 0).mean()), float(npred[s1mask].mean())))
            return rows

        out = {}
        for vi, vn in enumerate(variants):
            out[vn] = {}
            for a, src in enumerate(countries):
                for b, tgt in enumerate(countries):
                    if a == b:
                        continue
                    p = P[:, vi * len(countries) + a]
                    m_src, m_tgt = vmask & (cty_rows[:N1] == src), vmask & (cty_rows[:N1] == tgt)
                    cs, ct = curve(p, vc == src, m_src), curve(p, vc == tgt, m_tgt)
                    t_src, _, S_src, L_src = max(cs, key=lambda r: r[1])   # chosen on the SOURCE only
                    at = {r[0]: r for r in ct}
                    r_s = min(ct, key=lambda r: abs(r[2] - S_src)); r_l = min(ct, key=lambda r: abs(r[3] - L_src))
                    r_o = max(ct, key=lambda r: r[1])
                    rr = dict(t_src=t_src, f_src_t=at[t_src][1], t_singletons=r_s[0], f_singletons=r_s[1],
                              t_links=r_l[0], f_links=r_l[1], t_oracle=r_o[0], f_oracle=r_o[1])
                    ix = np.flatnonzero((vc == tgt) & (p >= P_FLOOR)); sel = ix[decode(si[ix], qi[ix], p[ix], 1, "thr", t_src)]
                    pred = np.zeros(len(p), bool); pred[sel] = True; tm = vc == tgt
                    for bn, bm in buckets.items():
                        m_ = tm & bm
                        rr[f"{bn}_prec"] = float((pred & m_ & (y == 1)).sum() / max((pred & m_).sum(), 1))
                        rr[f"{bn}_rec"] = float((pred & m_ & (y == 1)).sum() / max((m_ & (y == 1)).sum(), 1))
                    out[vn][f"{src}->{tgt}"] = rr
        res = dict(out["full"])
        dirs = list(out["full"])
        for k in dirs:
            f_, n_ = out["full"][k], out["nofrozen"][k]
            log("LCO", f"{k}: without frozen E5 {n_['f_src_t']:.4f} -> with {f_['f_src_t']:.4f} ({f_['f_src_t'] - n_['f_src_t']:+.4f})")
            for bn in buckets:
                log("LCO", f"   {k} {bn:10s}: precision {n_[bn + '_prec']:.4f} -> {f_[bn + '_prec']:.4f} | "
                           f"recall {n_[bn + '_rec']:.4f} -> {f_[bn + '_rec']:.4f}")
        deltas = [out["full"][k]["f_src_t"] - out["nofrozen"][k]["f_src_t"] for k in dirs]
        res["nofrozen"] = out["nofrozen"]
        res["frozen_deltas"] = dict(zip(dirs, deltas))
        res["frozen_ok"] = bool(all(d_ >= 0 for d_ in deltas))            # non-worse in EVERY direction
        res["frozen_convincing"] = bool(all(d_ >= 0.003 for d_ in deltas) and np.mean(deltas) >= 0.005)
        log("LCO", f"frozen E5 on unseen countries: deltas {[round(d_, 4) for d_ in deltas]} -> ok={res['frozen_ok']} "
                   f"convincing={res['frozen_convincing']}")
        pairs_ = [out["full"][k] for k in dirs]
        g_s = float(np.mean([v["f_singletons"] - v["f_src_t"] for v in pairs_]))
        g_l = float(np.mean([v["f_links"] - v["f_src_t"] for v in pairs_]))
        res.update(method="singletons" if g_s >= g_l else "links", gain=max(g_s, g_l), use=bool(max(g_s, g_l) > 0.0003))
        log("LCO", f"calibration by matched {res['method']}: mean gain {res['gain']:+.4f} over the source threshold -> use={res['use']}")
        json.dump(res, open(f"{CKPT}/lco_v49.json", "w"), indent=1)
    return res


LCO = lco_check()
M["frozen"] = dict(ok=LCO.get("frozen_ok", True), convincing=LCO.get("frozen_convincing", False), deltas=LCO.get("frozen_deltas"))
M["lco"] = LCO

In [ ]:
# Stage 5c (v4.6): domain-robust model for unseen countries — per-country quantile-normalised features
# Leave-country-out showed a model loses 0.02-0.06 on a country it never saw, and thresholds recover ~nothing.
# Here each similarity feature becomes "its percentile among this country's candidate pairs", which removes
# country-specific scale differences (e.g. long noisy Indian addresses vs short US ones). The same
# leave-country-out test decides whether France gets scores from this model.
ROBUST_COLS = ["cos", "gap_q", "marg_q", "gap_s", "name_ratio", "name_tset", "addr_tsort", "addr_tset", "len_ratio",
               "name_freq_s", "name_freq_q", "name2_freq_q", "addr_freq_s", "addr_freq_q", "core_tset", "core_ratio",
               "tf_name", "tf_core", "tf_addr", "tf_gap_q", "tf_gap_s"]
RIDX = [FEATS4.index(c) for c in ROBUST_COLS]
QGRID = np.linspace(0, 1, 201)


def qtables(c_df, rec, FQ, pair_cty, n=1_000_000):
    tabs = {}
    for c in sorted(set(pair_cty)):
        ix = np.sort(rng.permutation(np.flatnonzero(pair_cty == c))[:n])
        X = feats4(c_df.iloc[ix], rec, FQ, lambda X, ch: X)
        tabs[c] = np.quantile(X[:, RIDX], QGRID, axis=0) + np.arange(len(QGRID))[:, None] * 1e-7
    return tabs


def robust_x(X, pair_cty, tabs):
    X = X.copy()
    for c, q in tabs.items():
        m = pair_cty == c
        if m.any():
            for k, j in enumerate(RIDX):
                X[m, j] = np.interp(X[m, j], q[:, k], QGRID)
    return X


def lco_eval(P, countries, cty_rows):
    si, qi, y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
    vc = cty_rows[si]; res = {}
    for a, src in enumerate(countries):
        for b, tgt in enumerate(countries):
            if a == b:
                continue
            p = P[:, a]
            curves = []
            for cc_, m1 in ((src, vc == src), (tgt, vc == tgt)):
                ix = np.flatnonzero(m1 & (p >= P_FLOOR)); msk = vmask & (cty_rows[:N1] == cc_)
                rows = []
                for t in LCO_GRID:
                    sel = decode(si[ix], qi[ix], p[ix], 1, "thr", t)
                    rows.append((float(t), macro_f(si[ix][sel], y[ix][sel], n_true, msk)[0]))
                curves.append(rows)
            t_src = max(curves[0], key=lambda r: r[1])[0]
            res[f"{src}->{tgt}"] = dict(t_src=t_src, f_src_t=dict(curves[1])[t_src], f_oracle=max(r[1] for r in curves[1]))
    return res


SUBSET_DROP = ["cos", "marg_q", "gap_s", "addr_tsort", "addr_tset", "name_freq_s", "name_freq_q", "name2_freq_q",
               "addr_freq_s", "addr_freq_q", "len_ratio"]   # absolute, country-scaled signals
SDROP = [FEATS4.index(c) for c in SUBSET_DROP]


def subset_x(X):
    X = X.copy(); X[:, SDROP] = np.nan   # the model never sees these
    return X


def robust_transform(kind, X, pair_cty, tabs):
    return robust_x(X, pair_cty, tabs) if kind == "norm" else subset_x(X)


def robust_stage():
    """Two country-independent candidates, judged by leave-country-out on real labels:
    'norm' = per-country percentile features, 'subset' = model without absolute country-scaled features."""
    path, mpath = ck("robust_v46.json"), ck("xgb46_robust.json")
    if os.path.exists(path):
        info = json.load(open(path))
        return info, (xgb.Booster(model_file=mpath) if info.get("use") and os.path.exists(mpath) else None)
    cty_rows = tr.country.to_numpy(dtype=object)
    countries = sorted(set(cty_rows[:N1]))
    info, model = {"use": False}, None
    if len(countries) < 2 or not LCO:
        return info, model
    kinds = ("norm", "subset")
    with stage("ROBUST"):
        TT = qtables(cw, tr, FQ_tr, cty_rows[cw.si.values])
        cw_c = cty_rows[cw.si.values]; in_a2 = part[cw.si.values] == 1
        models = {}
        for c in countries:
            idx = np.sort(rng.permutation(np.flatnonzero(in_a2 & (cw_c == c)))[:LCO_PAIRS])
            samp = cw.iloc[idx]
            X = feats4(samp, tr, FQ_tr, lambda X, ch: X); yv = samp.y.to_numpy(); pc = cty_rows[samp.si.values]
            for kind in kinds:
                d = xgb.QuantileDMatrix(robust_transform(kind, X, pc, TT), yv)
                models[(kind, c)] = xgb.train(XGB_P, d, 800); del d; trim()
            del X, samp; trim()
            log("ROBUST", f"leave-country-out models trained on {c} | {mem()}")
        vfull = cw.loc[vsel]
        cols = [(k, c) for k in kinds for c in countries]
        P = feats4(vfull, tr, FQ_tr, lambda X, ch: np.column_stack(
            [models[(k, c)].predict(xgb.DMatrix(robust_transform(k, X, cty_rows[ch.si.values], TT))) for k, c in cols]).astype(np.float32))
        del vfull, models; trim()
        raw_f = float(np.mean([LCO[k_]["f_src_t"] for k_ in LCO if "->" in k_]))
        res = {}
        for i, kind in enumerate(kinds):
            r = lco_eval(P[:, i * len(countries):(i + 1) * len(countries)], countries, cty_rows)
            res[kind] = dict(lco=r, mean=float(np.mean([v["f_src_t"] for v in r.values()])))
            for k_, v in r.items():
                log("ROBUST", f"[{kind}] {k_}: raw {LCO[k_]['f_src_t']:.4f} -> {v['f_src_t']:.4f} (oracle {v['f_oracle']:.4f})")
        kind = max(kinds, key=lambda k: res[k]["mean"])
        info = dict(candidates=res, raw_mean=raw_f, kind=kind, gain=res[kind]["mean"] - raw_f,
                    use=bool(res[kind]["mean"] - raw_f > 0.002))
        log("ROBUST", f"unseen-country F0.5: raw {raw_f:.4f} | norm {res['norm']['mean']:.4f} | subset {res['subset']['mean']:.4f}"
                      f" -> best '{kind}' ({info['gain']:+.4f}) -> use for unseen countries={info['use']}")
        if info["use"]:
            idx = np.sort(rng.permutation(len(cw))[:XGB_MAX_PAIRS])
            samp = cw.iloc[idx]
            X = feats4(samp, tr, FQ_tr, lambda X, ch: robust_transform(kind, X, cty_rows[ch.si.values], TT))
            d = xgb.QuantileDMatrix(X, samp.y.to_numpy()); del X, samp; trim()
            model = xgb.train(XGB_P, d, 800); del d; trim()
            model.save_model(f"{CKPT}/xgb46_robust.json")
        json.dump(info, open(f"{CKPT}/robust_v46.json", "w"), indent=1)
    return info, model


ROBUST, ROBUST_MODEL = ({"use": False, "note": "v4.6b: normalised/subset models transferred worse than raw"}, None)
M["robust"] = ROBUST
val["conf"] = num_conflict(val.si.to_numpy(), val.qi.to_numpy(), FQ_tr)
del cw, p_oof, vS, vsel, FQ_tr
tr = tr[["entity_id", "src", "country", "txt", "ak"]]; trim()
log("VAL", f"{len(val):,} validation pairs ({int(vmask[val.si.values].sum()):,} with a V S1) | {mem()}")

with stage("DECODE-VAL"):
    sweep, best = tune_fast(val, [0])
    vo = val[vmask[val.si.values]]   # old style: owners of other S1s invisible, for comparison only
    _, best_old = tune_fast(vo.reset_index(drop=True), [0]); del vo
    M["decoder_xgb"] = best; M["decoder_xgb_v_only_old_style"] = best_old
    sweep.to_csv(f"{DIAG}/threshold_sweep.csv", index=False)
    log("DECODE", f"XGB, all owners visible: {best} | old V-only style: {best_old['f05']:.5f}")
dump()

In [ ]:
# Stage 6 (v4): test = saved candidates ∪ name-only channel, new features, SAFE submission
te = load_split("test")
N1T = int((te.src == 1).sum())
s_ok_te = np.zeros(len(te), bool); s_ok_te[:N1T] = True
q_ok_te = ~s_ok_te
if os.path.exists(ck("cand_test_v4.parquet")):
    tc = pd.read_parquet(ck("cand_test_v4.parquet"))
else:
    with stage("TEST-UNION"):
        tc0 = pd.read_parquet(ck("cand_test.parquet"), columns=["qi", "si", "d", "n", "a", "cos"])
        k_old = pair_key(tc0.si.values, tc0.qi.values)
        km = name_channel(te, N1T, s_ok_te, q_ok_te, "test")
        new = km[~np.isin(km, k_old)]
        nsi, nqi = new >> 32, new & 0xFFFFFFFF
        rows = np.unique(np.concatenate([nsi, nqi]))
        Ep = encode_texts(take(te.txt, rows), ck("emb_test_part.npy", big=True))
        cos_new = cos_pairs(Ep, nsi, nqi, pos=rows)
        keys = np.concatenate([k_old, new]); o = np.argsort(keys, kind="stable"); keys = keys[o]
        z = np.zeros(len(new), np.int8)
        tc = frame_from_keys(keys, {c_: np.concatenate([tc0[c_].to_numpy(), z])[o] for c_ in ("d", "n", "a")} | {"m": np.isin(keys, km)})
        tc["cos"] = np.concatenate([tc0.cos.to_numpy(), cos_new])[o]
        del tc0, k_old, keys, o, z; trim()
        tc = add_group_feats(tc)
        tc.to_parquet(ck("cand_test_v4.parquet"))
        log("TEST-UNION", f"{len(new):,} new name-channel pairs | {len(tc):,} total | {mem()}")
def _cos_test(a_, b_):
    rows_ = np.unique(np.concatenate([a_, b_]))
    Ep_ = encode_texts(take(te.txt, rows_), ck("emb_test_part48.npy", big=True))
    return cos_pairs(Ep_, a_, b_, pos=rows_)


tc = augment_exact(tc, te, N1T, s_ok_te, q_ok_te, "cand_test_v49.parquet", _cos_test)
_bk = pd.read_parquet(ck("cand_test_v4.parquet"), columns=["qi", "si"]) if os.path.exists(ck("cand_test_v4.parquet")) else None
tc["r"] = (~np.isin(pair_key(tc.si.values, tc.qi.values), pair_key(_bk.si.values, _bk.qi.values))).astype(np.int8) if _bk is not None else np.int8(0)
del _bk
tc = add_tf_cols(tc, te, "tf_test_v49.npz")
tc = add_frozen_cols(tc, te, "fz_test_v49.npz", "emb_frozen_test.npy")
FQ_te = freq_arrays(te, N1T, s_ok_te)
if os.path.exists(ck("test_p1_v49.npy")):
    tc["p1"] = np.load(ck("test_p1_v49.npy"))
else:
    with stage("XGB4-TEST"):
        tc["p1"] = feats4(tc, te, FQ_te, lambda X, ch: xm["full"].predict(xgb.DMatrix(X)).astype(np.float32))
        np.save(ck("test_p1_v49.npy"), tc.p1.to_numpy())

if ROBUST.get("use") and ROBUST_MODEL is not None:
    _cty_te = te.country.to_numpy(dtype=object)
    _seen = set(tr.country.to_numpy(dtype=object)[:N1])
    _unseen_pairs = np.flatnonzero(~np.isin(_cty_te[tc.si.to_numpy()], list(_seen)))
    if os.path.exists(ck("test_p1_v46.npy")):
        tc["p1"] = np.load(ck("test_p1_v46.npy"))
    elif len(_unseen_pairs):
        with stage("ROBUST-TEST"):
            _sub = tc.iloc[_unseen_pairs]
            _kind = ROBUST.get("kind", "norm")
            _tabs = qtables(_sub, te, FQ_te, _cty_te[_sub.si.values]) if _kind == "norm" else None
            _pr = feats4(_sub, te, FQ_te, lambda X, ch: ROBUST_MODEL.predict(
                xgb.DMatrix(robust_transform(_kind, X, _cty_te[ch.si.values], _tabs))).astype(np.float32))
            _p = tc.p1.to_numpy().copy()
            log("ROBUST-TEST", f"{len(_unseen_pairs):,} unseen-country pairs | mean score {_p[_unseen_pairs].mean():.4f} -> {_pr.mean():.4f} "
                               f"| share > 0.5: {(_p[_unseen_pairs] > 0.5).mean():.4f} -> {(_pr > 0.5).mean():.4f}")
            _p[_unseen_pairs] = _pr; tc["p1"] = _p
            np.save(f"{CKPT}/test_p1_v46.npy", _p)
            del _sub, _pr; trim()
if not M["frozen"]["ok"]:
    # frozen-E5 features hurt the unseen-country check: unseen countries keep the previous model's scores
    for _pp, _cp in (("test_p1_v48.npy", "cand_test_v48.parquet"), ("test_p1_v45.npy", "cand_test_v4.parquet")):
        if os.path.exists(ck(_pp)) and os.path.exists(ck(_cp)):
            _oc = pd.read_parquet(ck(_cp), columns=["qi", "si"]); _op = np.load(ck(_pp))
            if len(_op) != len(_oc):
                continue
            _ok = pair_key(_oc.si.values, _oc.qi.values); _o = np.argsort(_ok); _ok, _op = _ok[_o], _op[_o]
            _k = pair_key(tc.si.values, tc.qi.values)
            _pos = np.minimum(np.searchsorted(_ok, _k), len(_ok) - 1); _hit = _ok[_pos] == _k
            _un = ~np.isin(te.country.to_numpy(dtype=object)[tc.si.to_numpy()], list(set(tr.country.to_numpy(dtype=object)[:N1])))
            _p = tc.p1.to_numpy().copy(); _m = _un & _hit; _p[_m] = _op[_pos[_m]]; tc["p1"] = _p
            log("FROZEN", f"frozen features did not transfer: {int(_m.sum()):,} unseen-country pairs use scores from {_pp}")
            del _oc, _op, _ok, _k; trim()
            break
tc = tc[["qi", "si", "p1", "cos", "d", "n", "a", "m", "r"]]
trim()
M["test_candidates"] = dict(pairs=len(tc), name_channel_pairs=int(tc.m.sum()))
DKEY_TE = dup_keys(te, N1T)
write_submission(te, N1T, tc, tc.p1.to_numpy(), best, "xgb")
P_TEST = tc.p1.to_numpy()

In [ ]:
# Stage 7 (v4): cross-encoder on the ambiguous band (reuses the V3.1 model and its cached test scores)
def v31_ce_cache():
    """Rebuild (pair key -> score) from V3.1's position-keyed test shards, if they are attached."""
    dirs = [CKPT] + RESUME
    shards = {}
    for d in dirs:
        for f in glob.glob(f"{d}/ce_test_e5ce_*.npy"):
            shards[int(re.findall(r"_(\d+)\.npy$", f)[0])] = f
    if not shards or not all(os.path.exists(ck(f)) for f in ("cand_test.parquet", "test_p1.npy", "pairs_val.parquet")):
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    old = pd.read_parquet(ck("cand_test.parquet"), columns=["qi", "si"])
    a_old = amb(np.load(ck("test_p1.npy")))
    av = amb(pd.read_parquet(ck("pairs_val.parquet"), columns=["p1"]).p1.to_numpy())
    nv = max(1, int(0.08 * len(av)))
    cut_old = np.partition(av, nv - 1)[nv - 1]
    cr = np.flatnonzero(a_old <= cut_old); cr = cr[np.argsort(a_old[cr])]
    osi, oqi = old.si.to_numpy()[cr], old.qi.to_numpy()[cr]
    L = te.txt.str.len().to_numpy()
    order = np.argsort(L[osi] + L[oqi], kind="stable")
    step = 500_000
    if sorted(shards) != list(range(-(-len(cr) // step))):
        log("CE", f"V3.1 shard cache does not line up ({len(shards)} shards for {len(cr):,} pairs); not used")
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    keys, scores = [], []
    for j in sorted(shards):
        ix = order[j * step:(j + 1) * step]; s_ = np.load(shards[j])
        if len(s_) != len(ix):
            log("CE", "V3.1 shard length mismatch; cache not used")
            return np.zeros(0, np.int64), np.zeros(0, np.float32)
        keys.append(pair_key(osi[ix], oqi[ix])); scores.append(s_)
    keys, scores = np.concatenate(keys), np.concatenate(scores)
    chk = rng.choice(len(keys), min(2000, len(keys)), replace=False)
    fresh, _ = ce_score(ce_model, ce_tok, te, (keys[chk] >> 32), (keys[chk] & 0xFFFFFFFF))
    err = float(np.abs(fresh - scores[chk]).mean())
    if err > 0.02:
        log("CE", f"V3.1 cache failed the spot check (mean |diff| {err:.4f}); not used")
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    o = np.argsort(keys)
    log("CE", f"rebuilt {len(keys):,} cached test scores from V3.1 | spot check mean |diff| {err:.5f}")
    return keys[o], scores[o]


PCE_T = np.zeros(len(tc), np.float32); ROUTED_T = np.zeros(len(tc), bool)   # reranker scores on test (for stage 2)
CE_PT = ck("ce_e5ce.pt")
USE_CE = os.path.exists(CE_PT)
M["cross_encoder"] = f"{E5_MODEL} + 1-logit head (V3.1 weights)" if USE_CE else None
if not USE_CE:
    log("CE", "ce_e5ce.pt not attached; reranker stage skipped")
else:
    ce_tok, ce = load_ce(E5_MODEL)
    ce.load_state_dict(torch.load(CE_PT, map_location="cpu"))
    ce_model = par(Logit(ce)).eval()
    with stage("CE-VAL"):
        aV = amb(val.p1.to_numpy()); onV = vmask[val.si.values]
        nb = max(1, int(ROUTE_FRAC * onV.sum()))
        cut = np.partition(aV[onV], nb - 1)[nb - 1]
        vr = np.flatnonzero(aV <= cut)   # band pairs of V S1s and of their competitors
        pce_v, rate = ce_cached(tr, val.si.to_numpy()[vr], val.qi.to_numpy()[vr], "ce_val_cache_v42.npz", "ce_val_v42")
        val["routed"] = False; val["pce"] = 0.0
        val.loc[vr, "routed"] = True; val.loc[vr, "pce"] = pce_v
        sweep2, best2 = tune_fast2(val, [0, 0.25, 0.5, 0.75, 1.0])
        b_nv = sweep2[sweep2.veto == 0].iloc[0]; b_v = sweep2[sweep2.veto == 1].iloc[0] if (sweep2.veto == 1).any() else b_nv
        M["veto"] = dict(best_without=float(b_nv.f05), best_with=float(b_v.f05),
                         val_pairs_vetoed=int((val.routed & val.conf & (val.p1 < 0.5)).sum()))
        log("DECODE", f"number-conflict veto: best F0.5 without {b_nv.f05:.5f} | with {b_v.f05:.5f} | "
                      f"{M['veto']['val_pairs_vetoed']:,} routed validation pairs affected")
        if b_v.f05 >= b_nv.f05 - 1e-4 and not best2.get("veto"):
            # validation (US/India) rarely contains these overrides; the rule follows the stated hard-negative design
            # and matters most where the reranker overrides more (France: 4x the US rate). Prefer it unless it costs > 0.0001.
            best2 = b_v.to_dict(); log("DECODE", f"veto kept (costs {b_nv.f05 - b_v.f05:+.5f} on validation): {best2}")
        sweep = pd.concat([sweep, sweep2]).drop_duplicates().sort_values("f05", ascending=False)
        sweep.to_csv(f"{DIAG}/threshold_sweep.csv", index=False)
        M["decoder_ce"] = best2; M["ce_pairs_per_s"] = rate
        log("DECODE", f"best with reranker: {best2} | {len(vr):,} val pairs reranked at {rate:,.0f} pairs/s")
    if best2["w"] > 0 and best2["f05"] > best["f05"]:
        with stage("CE-TEST"):
            a = amb(tc.p1.to_numpy())
            cand_r = np.flatnonzero(a <= cut); cand_r = cand_r[np.argsort(a[cand_r])]
            kr = pair_key(tc.si.to_numpy()[cand_r], tc.qi.to_numpy()[cand_r])
            k31, s31 = v31_ce_cache(); k42, s42 = load_key_cache(["ce_test_cache_v42.npz"])
            ck_keys, ix_ = np.unique(np.concatenate([k42, k31]), return_index=True)   # own cache first on duplicates
            ck_scores = np.concatenate([s42, s31])[ix_]
            pce_t, hit = lookup(ck_keys, ck_scores, kr)
            miss = np.flatnonzero(~hit)
            budget = int(rate * max(0, left() - RESERVE_S))
            log("CE-TEST", f"routing {len(kr):,} test pairs | {hit.sum():,} cached | {len(miss):,} to score | budget {budget:,}")
            if len(miss) > budget:   # keep the most ambiguous ones if time is short
                drop = miss[budget:]; miss = miss[:budget]
                keep = np.ones(len(kr), bool); keep[drop] = False
                cand_r, kr, pce_t, hit = cand_r[keep], kr[keep], pce_t[keep], hit[keep]
                miss = np.flatnonzero(~hit)
            if len(miss):
                pce_t[miss], _ = ce_score(ce_model, ce_tok, te, tc.si.to_numpy()[cand_r[miss]], tc.qi.to_numpy()[cand_r[miss]], shard="ce_test_v42")
            np.savez(f"{CKPT}/ce_test_cache_v42.npz", keys=kr, scores=pce_t)
            p = tc.p1.to_numpy().copy()
            conf_t = num_conflict(tc.si.to_numpy()[cand_r], tc.qi.to_numpy()[cand_r], FQ_te)
            p[cand_r] = blend(p[cand_r], pce_t, np.ones(len(cand_r), bool), conf_t, best2["w"], int(best2.get("veto", 0) or 0))
            M["test_vetoed"] = int((conf_t & (tc.p1.to_numpy()[cand_r] < 0.5)).sum()) if best2.get("veto") else 0
            for c_ in sorted(set(te.country.to_numpy(dtype=object)[:N1T])):
                mc_ = te.country.to_numpy(dtype=object)[tc.si.to_numpy()[cand_r]] == c_
                log("CE-TEST", f"{c_}: {mc_.sum():,} routed | {(mc_ & conf_t & (tc.p1.to_numpy()[cand_r] < 0.5)).sum():,} reranker overrides with a number conflict")
            M["test_routed"] = int(len(cand_r))
            write_submission(te, N1T, tc, p, best2, "xgb+ce")
            P_TEST = p
            PCE_T[cand_r] = pce_t; ROUTED_T[cand_r] = True
        best = best2
    else:
        log("CE", "reranker did not improve validation F0.5; keeping the XGBoost submission")

In [ ]:
# Stage 7b2 (v4.5): duplicate consistency, kept only if validation improves
DKEY_TR = dup_keys(tr, N1)
with stage("DUP-CHECK"):
    _w = best.get("w", 0) or 0
    _pv = val.p1.to_numpy() if not _w else blend(val.p1.to_numpy(), val.pce.to_numpy(), val.routed.to_numpy(),
                                                 val.conf.to_numpy(), _w, int(best.get("veto", 0) or 0))
    _si, _qi, _y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
    _sel = decode(_si, _qi, _pv, best["uniq"], best["rule"], best["t"])
    f_off = macro_f(_si[_sel], _y[_sel], n_true, vmask)[0]
    _sel2 = dup_fill(_si, _qi, _pv, _sel, DKEY_TR)
    f_on = macro_f(_si[_sel2], _y[_sel2], n_true, vmask)[0]
    DUP = f_on > f_off + 1e-5
    M["dup_consistency"] = dict(f_without=f_off, f_with=f_on, added_links=int(len(_sel2) - len(_sel)), used=DUP)
    log("DUP", f"validation F0.5 without {f_off:.5f} | with {f_on:.5f} | +{len(_sel2) - len(_sel):,} links -> used={DUP}")
if DUP:
    write_submission(te, N1T, tc, P_TEST, best, "final+dup")

In [ ]:
# Stage 7b3 (v4.7): collective Stage-2 model — evidence from competing owners and already-confident siblings
# For each (S1 s, record q): how q's score compares with its other candidate owners, how many confident records s
# already has (from each source), whether q looks like those siblings, and duplicate-copy support. Features use
# only base scores (out-of-fold on validation), never labels. Trained on validation with S1-disjoint 2-fold OOF,
# kept only if it beats the base decoder by >= 0.0005.
FEATS2 = ["p", "p1", "pce", "routed", "margin_q", "rank_q", "cnt05_q", "n_q", "rank_s", "cnt_conf_s", "sum_p_s", "n_s",
          "other_src_conf", "dup_support", "sib_txt_max", "sib_num_eq", "sib_num_disjoint", "n_sib", "qs_txt_tset",
          "ak_empty_q", "is_s3"]


def collective_feats(si, qi, p, p1, pce, routed, rec, n1, dkey, num6, rows, step=3_000_000):
    N = len(rec); F = {}
    r_q, best_q, sec_q, n_q = grp(qi, p)
    F["margin_q"] = (p - np.where(r_q == 0, np.maximum(sec_q, 0), best_q))[rows]
    F["rank_q"], F["n_q"] = r_q[rows], n_q[rows]
    F["cnt05_q"] = np.bincount(qi[p > 0.5], minlength=N)[qi[rows]]
    conf = (r_q == 0) & (p >= 0.95); own = (r_q == 0) & (p >= 0.5)
    del best_q, sec_q, n_q
    r_s, _, _, n_s = grp(si, p)
    F["rank_s"], F["n_s"] = r_s[rows], n_s[rows]; del r_s, n_s
    F["cnt_conf_s"] = np.bincount(si[conf], minlength=n1)[si[rows]] - conf[rows]
    F["sum_p_s"] = np.bincount(si, weights=p, minlength=n1)[si[rows]] - p[rows]
    src = rec.src.to_numpy()[qi]
    c2 = np.bincount(si[conf & (src == 2)], minlength=n1); c3 = np.bincount(si[conf & (src == 3)], minlength=n1)
    F["other_src_conf"] = np.where(src[rows] == 2, c3[si[rows]], c2[si[rows]])
    dup = np.zeros(len(rows), np.float32)
    g = dkey[qi[rows]]; m = g != 0
    if m.any():
        ga = dkey[qi]; om = own & (ga != 0)
        a = pd.DataFrame({"g": ga[om], "s": si[om]}).value_counts().rename("c").reset_index()
        b = pd.DataFrame({"g": g[m], "s": si[rows][m], "i": np.flatnonzero(m), "self": own[rows][m].astype(np.int8)})
        b = b.merge(a, on=["g", "s"], how="left").fillna({"c": 0})
        dup[b.i.to_numpy()] = b.c.to_numpy() - b.self.to_numpy()
    F["dup_support"] = dup
    # up to 3 most confident sibling records per S1
    ci = np.flatnonzero(conf); ci = ci[np.lexsort((-p[ci], si[ci]))]
    ss = si[ci]
    start = np.r_[0, np.flatnonzero(ss[1:] != ss[:-1]) + 1] if len(ci) else np.zeros(0, np.int64)
    rk = np.arange(len(ci)) - np.repeat(start, np.diff(np.r_[start, len(ci)])) if len(ci) else np.zeros(0, np.int64)
    SIB = np.full((n1, 3), -1, np.int64); kk = rk < 3
    SIB[ss[kk], rk[kk]] = qi[ci[kk]]
    for f in ("sib_txt_max", "sib_num_eq", "sib_num_disjoint", "qs_txt_tset"):
        F[f] = np.full(len(rows), np.nan, np.float32)
    F["n_sib"] = np.zeros(len(rows), np.float32)
    band = np.flatnonzero(p[rows] < 0.98)          # confident pairs need no extra evidence
    for j in range(0, len(band), step):
        bi = band[j:j + step]; s_b, q_b = si[rows][bi], qi[rows][bi]
        qt = take(rec.txt, q_b)
        F["qs_txt_tset"][bi] = cpdist(qt, take(rec.txt, s_b), scorer=fuzz.token_set_ratio, workers=-1)
        A = num6[q_b]; best_sim = np.full(len(bi), -1.0, np.float32)
        any_eq = np.zeros(len(bi), bool); any_share = np.zeros(len(bi), bool); nsib = np.zeros(len(bi), np.float32)
        for k_ in range(3):
            sb = SIB[s_b, k_]; ok = np.flatnonzero((sb >= 0) & (sb != q_b))
            if not len(ok):
                continue
            best_sim[ok] = np.maximum(best_sim[ok], cpdist(qt[ok], take(rec.txt, sb[ok]), scorer=fuzz.token_set_ratio, workers=-1))
            B, Aa = num6[sb[ok]], A[ok]
            both = (Aa > 0).any(1) & (B > 0).any(1)
            share = ((Aa[:, :, None] == B[:, None, :]) & (Aa[:, :, None] > 0)).any(2).any(1)
            any_eq[ok] |= both & (Aa == B).all(1); any_share[ok] |= both & share; nsib[ok] += 1
        has = nsib > 0
        F["sib_txt_max"][bi] = np.where(has, best_sim, np.nan)
        F["sib_num_eq"][bi] = np.where(has, any_eq, np.nan)
        F["sib_num_disjoint"][bi] = np.where(has, ~any_share, np.nan)
        F["n_sib"][bi] = nsib
    F["p"], F["p1"], F["pce"], F["routed"] = p[rows], p1[rows], pce[rows], routed[rows]
    F["ak_empty_q"] = rec.ak.str.len().to_numpy()[qi[rows]] == 0
    F["is_s3"] = src[rows] == 3
    return np.column_stack([np.asarray(F[f], np.float32) for f in FEATS2])


COLL_USED, VAL_P2 = False, None
if left() > 2 * 3600:
    with stage("COLLECTIVE"):
        _w = best.get("w", 0) or 0; _veto = int(best.get("veto", 0) or 0)
        _pce_v = val.pce.to_numpy() if "pce" in val else np.zeros(len(val), np.float32)
        _rt_v = val.routed.to_numpy() if "routed" in val else np.zeros(len(val), bool)
        _conf_v = val.conf.to_numpy()
        pv = val.p1.to_numpy() if not _w else blend(val.p1.to_numpy(), _pce_v, _rt_v, _conf_v, _w, _veto)
        NUM6_TR = num_matrix(tr)
        vrows = np.flatnonzero(pv >= P_FLOOR)
        Xv = collective_feats(val.si.to_numpy(), val.qi.to_numpy(), pv, val.p1.to_numpy(), _pce_v, _rt_v.astype(np.float32),
                              tr, N1, DKEY_TR, NUM6_TR, vrows)
        yv = val.y.to_numpy()[vrows]; fold = val.si.to_numpy()[vrows] % 2
        P2_PARAMS = dict(XGB_P, max_depth=6)
        p2 = pv.copy(); iters2 = []
        for f in (0, 1):
            dtr = xgb.QuantileDMatrix(Xv[fold != f], yv[fold != f]); dva = xgb.QuantileDMatrix(Xv[fold == f], yv[fold == f], ref=dtr)
            b2 = xgb.train(P2_PARAMS, dtr, 1500, evals=[(dva, "val")], early_stopping_rounds=50, verbose_eval=300)
            iters2.append(b2.best_iteration + 1)
            p2[vrows[fold == f]] = b2[: b2.best_iteration + 1].predict(xgb.DMatrix(Xv[fold == f]))
            del dtr, dva, b2; trim()
        _si, _qi, _y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
        _s = decode(_si, _qi, pv, best["uniq"], best["rule"], best["t"]); f_base = macro_f(_si[_s], _y[_s], n_true, vmask)[0]
        curve2 = []
        for t in np.round(np.arange(0.2, 0.951, 0.025), 3):
            _s = decode(_si, _qi, p2, 1, "thr", t); curve2.append((float(t), macro_f(_si[_s], _y[_s], n_true, vmask)[0]))
        t2, f2 = max(curve2, key=lambda r: r[1])
        COLL_USED = f2 > f_base + 0.0005
        M["collective"] = dict(f_base=f_base, f_stage2=f2, t=t2, used=COLL_USED, iters=iters2)
        log("COLLECTIVE", f"validation F0.5: base {f_base:.5f} -> stage-2 {f2:.5f} ({f2 - f_base:+.5f}) at t={t2} -> used={COLL_USED}")
        # Stage-2 on an unseen country only if it transfers: train on one country's rows, score the other
        COLL_FR_OK = False
        if COLL_USED:
            _cty = tr.country.to_numpy(dtype=object)
            _vc = _cty[_si[vrows]]; _cs = sorted(set(_cty[:N1]))
            lco2, ok_all = {}, len(_cs) >= 2
            for a_ in _cs:
                for b_ in _cs:
                    if a_ == b_:
                        continue
                    ma, mb = _vc == a_, _vc == b_
                    mdl = xgb.train(P2_PARAMS, xgb.QuantileDMatrix(Xv[ma], yv[ma]), int(np.mean(iters2)))
                    p_l = pv.copy(); p_l[vrows[mb]] = mdl.predict(xgb.DMatrix(Xv[mb]))
                    msk_a, msk_b = vmask & (_cty[:N1] == a_), vmask & (_cty[:N1] == b_)
                    pa = np.flatnonzero(_cty[_si] == a_); pb = np.flatnonzero(_cty[_si] == b_)
                    # threshold chosen ONLY from the source country: source-only 2-fold out-of-fold stage-2 scores
                    p_src = pv.copy(); ra = vrows[ma]; fa = _si[ra] % 2
                    for f_ in (0, 1):
                        mf = xgb.train(P2_PARAMS, xgb.QuantileDMatrix(Xv[ma][fa != f_], yv[ma][fa != f_]), int(np.mean(iters2)))
                        p_src[ra[fa == f_]] = mf.predict(xgb.DMatrix(Xv[ma][fa == f_])); del mf
                    ta = max(((t, macro_f(_si[pa][s_], _y[pa][s_], n_true, msk_a)[0]) for t in np.round(np.arange(0.2, 0.951, 0.025), 3)
                              for s_ in [decode(_si[pa], _qi[pa], p_src[pa], 1, "thr", t)]), key=lambda r: r[1])[0]
                    s2 = decode(_si[pb], _qi[pb], p_l[pb], 1, "thr", ta); f_s2 = macro_f(_si[pb][s2], _y[pb][s2], n_true, msk_b)[0]
                    s0 = decode(_si[pb], _qi[pb], pv[pb], best["uniq"], best["rule"], best["t"]); f_b0 = macro_f(_si[pb][s0], _y[pb][s0], n_true, msk_b)[0]
                    lco2[f"{a_}->{b_}"] = dict(base=f_b0, stage2=f_s2, t=ta)
                    ok_all &= f_s2 >= f_b0 - 0.0005
                    log("COLLECTIVE-LCO", f"{a_}->{b_}: base {f_b0:.5f} -> stage-2 trained on {a_} only {f_s2:.5f} ({f_s2 - f_b0:+.5f})")
                    del mdl; trim()
            COLL_FR_OK = bool(ok_all)
            M["collective"].update(lco=lco2, france_ok=COLL_FR_OK)
            log("COLLECTIVE", f"stage-2 for unseen countries: {'YES' if COLL_FR_OK else 'NO (keep the v4.5 France path)'}")
        if COLL_USED:
            m2 = xgb.train(P2_PARAMS, xgb.QuantileDMatrix(Xv, yv), int(np.mean(iters2) * 1.1))
            imp = m2.get_score(importance_type="gain")
            M["collective"]["gain"] = {FEATS2[int(k[1:])]: round(v, 1) for k, v in sorted(imp.items(), key=lambda t: -t[1])[:12]}
            log("COLLECTIVE", f"top features: {list(M['collective']['gain'])[:8]}")
            trows = np.flatnonzero(P_TEST >= P_FLOOR)
            Xt = collective_feats(tc.si.to_numpy(), tc.qi.to_numpy(), P_TEST, tc.p1.to_numpy(), PCE_T, ROUTED_T.astype(np.float32),
                                  te, N1T, DKEY_TE, FQ_te["num6"], trows)
            P_BASE_T = P_TEST.copy()
            P_S2_ALL = P_TEST.copy(); P_S2_ALL[trows] = m2.predict(xgb.DMatrix(Xt)).astype(np.float32)
            _unseen_t = ~np.isin(te.country.to_numpy(dtype=object)[tc.si.to_numpy()], list(set(tr.country.to_numpy(dtype=object)[:N1])))
            P_S2_SEEN = np.where(_unseen_t, P_BASE_T, P_S2_ALL).astype(np.float32)
            P_TEST = P_S2_ALL if COLL_FR_OK else P_S2_SEEN
            P_S2_OTHER = (P_S2_SEEN, "stage2_seen_only") if COLL_FR_OK else (P_S2_ALL, "stage2_all")
            VAL_P2 = p2
            log("COLLECTIVE", f"main file: stage-2 {'everywhere' if COLL_FR_OK else 'for seen countries, v4.5 scores for unseen'}; "
                              f"other version -> variants/matching_results_{P_S2_OTHER[1]}.tsv")
            best = dict(best, uniq=1, rule="thr", t=t2, f05=f2, stage2=1)
            write_submission(te, N1T, tc, P_TEST, best, "stage2")
            del Xt
        del Xv, NUM6_TR; trim()
    dump()
else:
    log("COLLECTIVE", "skipped: less than 2 h budget left")

In [ ]:
# Stage 7b4 (v4.9): does the empty-address rescue help on validation? (switch it off at decode level if not)
RESCUE_ON = True
if int(val.r.sum()) > 0:
    with stage("RESCUE-CHECK"):
        _w = best.get("w", 0) or 0
        _pv = VAL_P2 if VAL_P2 is not None else (val.p1.to_numpy() if not _w else blend(
            val.p1.to_numpy(), val.pce.to_numpy(), val.routed.to_numpy(), val.conf.to_numpy(), _w, int(best.get("veto", 0) or 0)))
        _si, _qi, _y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
        _s = decode(_si, _qi, _pv, best["uniq"], best["rule"], best["t"]); f_on = macro_f(_si[_s], _y[_s], n_true, vmask)[0]
        _pv_off = np.where(val.r.to_numpy() == 1, 0, _pv).astype(np.float32)
        _s = decode(_si, _qi, _pv_off, best["uniq"], best["rule"], best["t"]); f_off = macro_f(_si[_s], _y[_s], n_true, vmask)[0]
        RESCUE_ON = f_on >= f_off
        M["rescue"] = dict(f_on=f_on, f_off=f_off, used=RESCUE_ON, val_pairs=int(val.r.sum()), test_pairs=int(tc.r.sum()))
        log("RESCUE", f"validation F0.5 with rescued pairs {f_on:.5f} | without {f_off:.5f} -> keep rescue={RESCUE_ON}")
        if not RESCUE_ON:
            P_TEST = np.where(tc.r.to_numpy() == 1, 0, P_TEST).astype(np.float32); VAL_P2 = _pv_off
            write_submission(te, N1T, tc, P_TEST, best, "no-rescue")

In [ ]:
# Stage 7c: threshold variants of the final submission (same scores, same candidates)
VARIANT_T = [float(x) for x in os.environ.get("ER_VARIANTS", "0.55,0.6,0.7,0.75,0.8,0.85,0.9").split(",") if x]
os.makedirs(f"{OUT}/variants", exist_ok=True)
for _t in VARIANT_T:   # always global-threshold variants, whatever rule was chosen
    _sel = decode_final(tc.si.to_numpy(), tc.qi.to_numpy(), P_TEST, best["uniq"], "thr", _t, DKEY_TE)
    _path = f"{OUT}/variants/matching_results_t{_t:.2f}.tsv"
    write_lists(te, N1T, tc.si.to_numpy()[_sel], tc.qi.to_numpy()[_sel], _path, "matched_entity_ids")
    _np = np.bincount(tc.si.to_numpy()[_sel], minlength=N1T)
    _vs = sweep.query(f"w == {best['w']} and uniq == {best['uniq']} and rule == 'thr'") if not best.get("stage2") else sweep.iloc[:0]
    if "veto" in _vs and best.get("veto") == best.get("veto"):   # not NaN
        _vs = _vs[_vs.veto == best.get("veto")]
    _vf = _vs.loc[(_vs.t - _t).abs() < 1e-6, "f05"]
    log("VARIANT", f"t={_t:.2f}: {len(_sel):,} links | singletons {(_np == 0).mean():.3%} | val F0.5 "
                   f"{float(_vf.iloc[0]) if len(_vf) else float('nan'):.5f} | {_path}")

In [ ]:
# Stage 7d (v4.3): unseen-country calibration (France) from the generator constants
# Train shows identical singleton rate / matches per S1 in every country, so an unseen country should look like
# the seen ones at the same operating point. Its threshold is set so its predicted singletons (or links per S1)
# match the seen countries' TEST predictions.
test_cty = te.country.to_numpy(dtype=object); s1_cty = test_cty[:N1T]
seen = set(tr.country.to_numpy(dtype=object)[:N1])
si_t, qi_t = tc.si.to_numpy(), tc.qi.to_numpy(); c_pair = test_cty[si_t]
countries_t = sorted(set(s1_cty))
live = P_TEST >= P_FLOOR


def country_decode(t_map, P=None):
    P = P_TEST if P is None else P
    out = []
    for c in countries_t:
        ix = np.flatnonzero((c_pair == c) & (P >= P_FLOOR))
        out.append(ix[decode_final(si_t[ix], qi_t[ix], P[ix], best["uniq"], "thr", t_map[c], DKEY_TE)])
    return np.concatenate(out)


def calibrate_t(c, P, key, target):
    ix = np.flatnonzero((c_pair == c) & (P >= P_FLOOR)); rows = []
    for t in LCO_GRID:
        npred = np.bincount(si_t[ix][decode(si_t[ix], qi_t[ix], P[ix], best["uniq"], "thr", t)], minlength=N1T)[s1_cty == c]
        rows.append((float(t), float((npred == 0).mean()), float(npred.mean())))
    return min(rows, key=lambda r: abs(r[key] - target))[0]


def country_stats(sel):
    npred = np.bincount(si_t[sel], minlength=N1T)
    return {c: (float((npred[s1_cty == c] == 0).mean()), float(npred[s1_cty == c].mean())) for c in countries_t}


def quick_validate(path):
    if VALIDATOR:
        r = subprocess.run([sys.executable, VALIDATOR, "--matching", path, "--candidate", f"{OUT}/_skip.tsv",
                            "--test-dir", os.path.dirname(TSVS["test_source1.tsv"])], capture_output=True, text=True)
        log("VALIDATE", (r.stdout + r.stderr)[-600:])


if best["rule"] == "thr":
    with stage("COUNTRY-CAL"):
        t0 = float(best["t"])
        st = country_stats(country_decode({c: t0 for c in countries_t}))
        for c in countries_t:
            log("COUNTRY", f"{c:8s} at t={t0}: singletons {st[c][0]:.4f} | links/S1 {st[c][1]:.3f}")
        seen_t = [c for c in countries_t if c in seen]; unseen_t = [c for c in countries_t if c not in seen]
        if seen_t and unseen_t:
            key = 1 if LCO.get("method", "singletons") == "singletons" else 2
            target = float(np.mean([st[c][key - 1] for c in seen_t]))
            t_map = {c: t0 for c in countries_t}
            for c in unseen_t:
                t_map[c] = calibrate_t(c, P_TEST, key, target)
                log("COUNTRY", f"{c}: calibrated t={t_map[c]} (target {'singletons' if key == 1 else 'links/S1'} {target:.4f})")
            cal = country_decode(t_map); stc = country_stats(cal)
            for c in countries_t:
                log("COUNTRY", f"calibrated {c:8s}: singletons {stc[c][0]:.4f} | links/S1 {stc[c][1]:.3f} | t={t_map[c]}")
            M["country_calibration"] = dict(t0=t0, t_map=t_map, before=st, after=stc, lco_use=LCO.get("use", False))
            write_lists(te, N1T, si_t[cal], qi_t[cal], f"{OUT}/variants/matching_results_calibrated.tsv", "matched_entity_ids")
            for c in unseen_t:
                for dt in (-0.1, 0.1):
                    tv = round(min(max(t_map[c] + dt, 0.2), 0.95), 3)
                    sv = country_decode({**t_map, c: tv})
                    write_lists(te, N1T, si_t[sv], qi_t[sv], f"{OUT}/variants/matching_results_{c}_t{tv:.2f}.tsv", "matched_entity_ids")
                    log("VARIANT", f"{c} t={tv:.2f} (others t={t0}): {len(sv):,} links")
            # unseen countries scored by XGB alone (the reranker overrides 4x more often on French text)
            P_alt = np.where(np.isin(c_pair, unseen_t), tc.p1.to_numpy(), P_TEST).astype(np.float32)
            t_alt = dict(t_map)
            for c in unseen_t:
                t_alt[c] = calibrate_t(c, P_alt, key, target)
            sa = country_decode(t_alt, P_alt); sta = country_stats(sa)
            write_lists(te, N1T, si_t[sa], qi_t[sa], f"{OUT}/variants/matching_results_unseen_noCE.tsv", "matched_entity_ids")
            for c in unseen_t:
                log("VARIANT", f"{c} without reranker: t={t_alt[c]} | singletons {sta[c][0]:.4f} | links/S1 {sta[c][1]:.3f} "
                               f"-> variants/matching_results_unseen_noCE.tsv")
            M["country_calibration"]["t_noCE"] = t_alt
            if COLL_USED and "P_S2_OTHER" in globals():
                P_o, name_o = P_S2_OTHER
                t_o = dict(t_map)
                for c in unseen_t:
                    t_o[c] = calibrate_t(c, P_o, key, target)
                so = country_decode(t_o, P_o); sto = country_stats(so)
                write_lists(te, N1T, si_t[so], qi_t[so], f"{OUT}/variants/matching_results_{name_o}.tsv", "matched_entity_ids")
                for c in unseen_t:
                    log("VARIANT", f"{name_o}: {c} t={t_o[c]} | singletons {sto[c][0]:.4f} | links/S1 {sto[c][1]:.3f}")
            if LCO.get("use", False) and any(t_map[c] != t0 for c in unseen_t):
                shutil.copy(f"{OUT}/matching_results.tsv", f"{OUT}/variants/matching_results_uncalibrated.tsv")
                write_lists(te, N1T, si_t[cal], qi_t[cal], f"{OUT}/matching_results.tsv", "matched_entity_ids")
                log("SUBMIT", f"[calibrated] main file uses per-country thresholds {t_map}")
                quick_validate(f"{OUT}/matching_results.tsv")
            else:
                log("SUBMIT", "calibration not applied to the main file (leave-country-out check did not support it); "
                              "see output/variants/matching_results_calibrated.tsv")
        dump()

In [ ]:
# Stage 8: validation diagnostics, error attribution, artifacts
with stage("DIAG"):
    si, qi, y = val.si.to_numpy(), val.qi.to_numpy(), val.y.to_numpy()
    w = best["w"]
    p = val.p1.to_numpy() if w == 0 else blend(val.p1.to_numpy(), val.pce.to_numpy(), val.routed.to_numpy(), val.conf.to_numpy(), w, int(best.get("veto", 0) or 0))
    if VAL_P2 is not None:
        p = VAL_P2
    sel = decode_final(si, qi, p, best["uniq"], best["rule"], best["t"], DKEY_TR)
    f_all, f = macro_f(si[sel], y[sel], n_true, vmask)
    tp = np.bincount(si[sel], weights=y[sel], minlength=N1); npred = np.bincount(si[sel], minlength=N1)
    nt_v = n_true * vmask
    M["final_val"] = dict(macro_f05=f_all, precision=tp[vmask].sum() / max(npred[vmask].sum(), 1), recall=tp[vmask].sum() / nt_v.sum(),
                          singleton_f05=float(f[vmask & (n_true == 0)].mean()), non_singleton_f05=float(f[vmask & (n_true > 0)].mean()))
    cty = tr.country.to_numpy(dtype=object)[:N1]
    crow = [dict(country=c, s1=int((vmask & (cty == c)).sum()), f05=float(f[vmask & (cty == c)].mean())) for c in np.unique(cty[vmask])]
    # leave-country-out threshold robustness: tune the rule on one country, score on the others
    for c in np.unique(cty[vmask]):
        grid = np.arange(0.1, 0.96, 0.05) if best["rule"] == "thr" else (0.5, 0.75, 1, 1.5, 2, 3)
        sc = [(macro_f(si[s_], y[s_], n_true, vmask & (cty == c))[0], t) for t in grid for s_ in [decode(si, qi, p, best["uniq"], best["rule"], t)]]
        t_c = max(sc)[1]; s_ = decode(si, qi, p, best["uniq"], best["rule"], t_c)
        for r in crow:
            if r["country"] != c:
                r[f"f05_tuned_on_{c}"] = macro_f(si[s_], y[s_], n_true, vmask & (cty == r["country"]))[0]
    pd.DataFrame(crow).to_csv(f"{DIAG}/country_metrics.csv", index=False)

    # pair-level error attribution
    pred = np.zeros(len(val), bool); pred[sel] = True
    owner = np.full(len(tr), -1, np.int64); owner[t_qi] = t_si
    q_nonascii = ~pd.Series(take(tr.txt, qi)).map(str.isascii).to_numpy()
    q_noaddr = tr.ak.str.len().to_numpy()[qi] == 0
    onv = vmask[si]   # errors are counted on V S1s only; competitor pairs of other S1s are context
    fp, fn = pred & (y == 0) & onv, ~pred & (y == 1) & onv
    tv_miss = int((~found).sum())
    att = {"blocking_missed_links": tv_miss,
           "false_merge_on_singleton": int((fp & (n_true[si] == 0)).sum()),
           "false_merge_conflict(q belongs to another S1)": int((fp & (owner[qi] >= 0)).sum()),
           "false_merge_distractor(q belongs to no S1)": int((fp & (owner[qi] < 0) & (n_true[si] > 0)).sum()),
           "fp_matcher(p>=0.9)": int((fp & (p >= 0.9)).sum()), "fp_decoder(p<0.9)": int((fp & (p < 0.9)).sum()),
           "fn_matcher(p<0.1)": int((fn & (p < 0.1)).sum()), "fn_decoder(p>=0.1)": int((fn & (p >= 0.1)).sum()),
           "fp_non_ascii": int((fp & q_nonascii).sum()), "fn_non_ascii": int((fn & q_nonascii).sum()),
           "fp_empty_addr": int((fp & q_noaddr).sum()), "fn_empty_addr": int((fn & q_noaddr).sum())}
    M["error_attribution"] = att
    def err_tsv(mask, path, n=3000):
        ix = np.flatnonzero(mask)[:n]
        pd.DataFrame({"s1_id": take(tr.entity_id, si[ix]), "s1": take(tr.txt, si[ix]), "q_id": take(tr.entity_id, qi[ix]),
                      "q": take(tr.txt, qi[ix]), "p": p[ix], "cos": val.cos.to_numpy()[ix]}).to_csv(path, sep="\t", index=False)
    err_tsv(fp, f"{DIAG}/false_merges.tsv"); err_tsv(fn, f"{DIAG}/missed_matches.tsv"); err_tsv(fp | fn, f"{DIAG}/error_samples.tsv", 5000)

    # ablations from cached validation scores (no extra inference)
    abl = {"blocking_oracle": M["blocking_world"]["oracle_macro_f05"], "xgb_v_only_old_style": M["decoder_xgb_v_only_old_style"]["f05"],
           "xgb_best": M["decoder_xgb"]["f05"]}
    for name, q_ in [("xgb_thr_no_uniq", "w==0 and uniq==0 and rule=='thr'"), ("xgb_thr_uniq", "w==0 and uniq==1 and rule=='thr'"),
                     ("xgb_efb_uniq", "w==0 and uniq==1 and rule=='efb'"), ("with_ce_best", "w>0")]:
        s_ = sweep.query(q_)
        if len(s_):
            abl[name] = float(s_.f05.max())
    M["ablation"] = abl
    pd.DataFrame([dict(stage=k, f05=v) for k, v in abl.items()]).to_csv(f"{DIAG}/stage_metrics.csv", index=False)
    dump()

    print("\n" + "=" * 60)
    fv = M["final_val"]
    print(f"BEST VALIDATION MACRO F0.5: {fv['macro_f05']:.5f}\n")
    print(f"Blocking recall / ceiling:  {M['blocking_world']['recall']:.5f} / {M['blocking_world']['oracle_macro_f05']:.5f}")
    print(f"Final precision / recall:   {fv['precision']:.5f} / {fv['recall']:.5f}")
    print(f"Singleton / non-singleton:  {fv['singleton_f05']:.5f} / {fv['non_singleton_f05']:.5f}")
    for r in crow:
        print(f"  {r['country']}: {r['f05']:.5f}  " + "  ".join(f"{k}={v:.5f}" for k, v in r.items() if k.startswith("f05_tuned")))
    print("\nFailure attribution (validation pairs):")
    for k_, v_ in att.items():
        print(f"  {k_:48s} {v_:,}")
    print("\nAblation:", json.dumps(abl, indent=1))
    print("Decoder chosen:", best)
    print("=" * 60)

# write the artifacts zip FIRST, so a problem in the final check can never cost the outputs
import zipfile
with zipfile.ZipFile(f"{WORK}/final_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, DIAG):
        for f in glob.glob(f"{d}/*"):
            z.write(f, os.path.relpath(f, WORK))
# free every large object before the validator subprocess (v4.6b died here: parent held 17 GB)
for _v in ("tr", "te", "tc", "val", "a2", "cand", "cv", "hit", "e5", "ce", "ce_model", "FQ_te", "FQ_tr", "cw", "_e5_frozen"):
    globals().pop(_v, None)
for _k, _o in list(globals().items()):
    if not _k.startswith("__") and isinstance(_o, (np.ndarray, pd.DataFrame, pd.Series)):
        _nb = _o.nbytes if isinstance(_o, np.ndarray) else int(_o.memory_usage(deep=False).sum() if isinstance(_o, pd.DataFrame) else _o.memory_usage(deep=False))
        if _nb > 50_000_000:
            globals().pop(_k, None)
trim(); torch.cuda.empty_cache()
if VALIDATOR:
    _avail = int(dict(l.split(":", 1) for l in open("/proc/meminfo"))["MemAvailable"].split()[0]) / 1e6
    _full = _avail > 24   # the candidate cross-check alone needs ~10-14 GB (it loads ~100M IDs into Python sets)
    log("VALIDATE", f"final check ({'matching + candidates' if _full else 'matching only; candidates are a superset by construction'}) | {mem()}")
    r = subprocess.run([sys.executable, VALIDATOR, "--matching", f"{OUT}/matching_results.tsv",
                        "--candidate", f"{OUT}/candidate_pairs.tsv" if _full else f"{OUT}/_skip.tsv", "--check-ids",
                        "--test-dir", os.path.dirname(TSVS["test_source1.tsv"])], capture_output=True, text=True)
    log("VALIDATE", (r.stdout + r.stderr)[-3000:])
    M["validator_final"] = ("PASS" if r.returncode == 0 else f"FAIL (exit {r.returncode})") + ("" if _full else " (matching only)")
    dump()

log("DONE", f"total {(time.time() - T0) / 3600:.2f} h | outputs in {OUT}, artifacts in {WORK}/final_artifacts.zip")